# Run Plan Disclosure Pacing: Diary Data Pipeline (MCO2 Milestone 1)

**HCI2000 · Bernandino, Galman, Muros, Punsalan**

### Research questions
| RQ | Question | Main measures |
|---|---|---|
| **RQ1** | How does disclosure pacing (Full Plan vs Current Only) affect runners' cognitive load? | NASA-TLX (mental demand as the main indicator), diary text on what runners had to keep track of |
| **RQ2** | How does disclosure pacing affect runners' perceived exertion? | Borg RPE 6–20, diary text on pacing decisions |
| **RQ3** | How is cognitive load related to perceived exertion, and does this differ by condition? | Mental demand × RPE (within and between runners) |

### Expected relationships (Section 3.6 of the proposal; explored, not tested as confirmatory hypotheses)
1. **Cognitive load:** Full Plan > Current Only, mainly on mental demand.
2. **Perceived exertion:** RPE differs between conditions (non-directional). Anticipatory regulation predicts Full Plan runners hold back before hard sessions and push before easy ones or the final session.
3. **Load and exertion:** higher mental demand goes with higher RPE; the relationship may differ by condition.

### Pipeline overview
| Stage | Section | Output |
|---|---|---|
| Load + schema check | 1 | raw responses mapped to short field names |
| Cleaning (11 checks) | 2 | `cleaning_log.csv`, cleaned rows |
| Preparation | 3 | `sessions.csv` (one row per participant × scheduled session), `tlx_long.csv`, `participant_summary.csv`, `text_responses.csv`, `data_dictionary.csv` |
| Pipeline tests | 4 | unit tests + known-answer tests against the synthetic ground truth |
| Analysis | 5–9 | adherence, RQ1, RQ2, RQ3, LLM-assisted content analysis, sensitivity checks |

**To run:** put the two CSVs in `data/` (`diary_responses.csv` = the Google Forms export, `participant_key.csv` = the assignment key; both are git-ignored and must never be pushed), then *Run all*. Works in Jupyter, VS Code, or Google Colab (upload the `data/` folder). For the real study, change `RAW_FILE` and `KEY_FILE` in Section 0; `N_WEEKS` switches to 2 automatically when the file name does not contain `SYNTHETIC`.

## 0. Setup and study settings
Every study-specific assumption lives in this one cell, so switching from synthetic to real data means editing settings, not code.

In [ ]:
import itertools
import json
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import stats
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

# ---------- Files ----------
DATA_DIR = Path("data")
RAW_FILE = DATA_DIR / "diary_responses.csv"   # Google Forms export (File > Download > CSV)
KEY_FILE = DATA_DIR / "participant_key.csv"   # participant_code, condition, experience_group
OUT_DIR = Path("processed")
FIG_DIR = OUT_DIR / "figures"
LLM_DIR = OUT_DIR / "llm_coding"
for d in (OUT_DIR, FIG_DIR, LLM_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------- Study design ----------
STUDY_START = pd.Timestamp("2026-11-09")      # Monday of Week 1
IS_SYNTHETIC = "SYNTHETIC" in RAW_FILE.name
N_WEEKS = 2   
PLAN = ["Easy run", "Interval run", "Easy run", "Tempo run", "Long run"]   # session order within each week
SESSIONS_PER_WEEK = len(PLAN)
HARD_TYPES = {"Interval run", "Tempo run", "Long run"}
CONDITIONS = ["Full Plan", "Current Only"]

# ---------- Cleaning rules ----------
LATE_THRESHOLD_H = 2.0          # the form asks for entries within 2 hours of finishing
VERY_LATE_H = 24.0              # beyond this, recall is unreliable
DUPLICATE_POLICY = "last"       # "last" = treat a resubmission as a correction; "first" = keep the original
STOPPED_EARLY_RUN = 3           # this many unlogged sessions in a row at the end = "stopped logging early".
                                #   Flag only: those sessions stay "missing", because we do not know whether the
                                #   runner kept running without logging. It is NOT treated as a withdrawal.
CONFIRMED_WITHDRAWALS = {}      # only runners who TOLD the researchers they withdrew, e.g. {"R03": 8} = after session index 8
                                #   Runners with no entries at all are "did_not_start" (enrolled, never logged).
MIN_SUBMISSION_RATE = 0.5       # proposal's exclusion rule (fewer than 50% of diary entries). NOT applied to the main
                                #   analysis (sample too small; reported as a protocol deviation). Section 11 shows
                                #   the results with these runners excluded.
CONFIRMED_DATES = {("R05", 1, 5)}   # (code, week, session) whose date the runner confirmed: the session was run late,
                                #   in the following week, so C7 does not flag it.
MANUAL_CORRECTIONS = {}         # fixes confirmed with the runner, keyed by (code as submitted, Google Forms timestamp):
                                #   {("R03", "11/11/2026 7:50:16"): {"participant_code": "R08"}}   wrong code picked
                                #   {("R05", "11/12/2026 18:02:41"): {"week": 1, "session": 3}}    wrong session picked
                                #   {("R01", "11/9/2026 6:12:00"): {"drop": "researcher test"}}    not a real entry
TIMESTAMP_FORMAT = "%m/%d/%Y %H:%M:%S"   # Google Forms export formats; C2 stops with a clear error if they differ
DATE_FORMAT = "%m/%d/%Y"
TIME_FORMAT = "%I:%M:%S %p"
DATA_WITHDRAWN = set()          # participants who asked for their data to be removed (dropped entirely)
PERFORMANCE_REVERSED = False    # True if the form's Performance scale has 0 = failure (NASA-TLX standard is 0 = perfect)
SEED = 2000
rng = np.random.default_rng(SEED)

# ---------- Chart style (validated categorical palette, colorblind-safe for 2-3 series) ----------
COND_COLORS = {"Full Plan": "#2a78d6", "Current Only": "#eb6834"}
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
STATUS_COLORS = {"prescribed": "#0ca30c", "modified": "#fab219", "skipped": "#d03b3b",
                 "missing": "#e1e0d9", "withdrawn": "#f0efec", "did_not_start": "#f7f6f2"}
mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "grid.linestyle": "-", "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left", "font.size": 10,
    "axes.labelsize": 10, "legend.frameon": False, "figure.dpi": 110, "axes.axisbelow": True,
})

def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")

print("Settings loaded. Expected sessions per participant:", N_WEEKS * SESSIONS_PER_WEEK)

## 1. Load raw data and check the schema
Google Forms exports use the full question text as column headers, and editing a question changes the header. Instead of hard-coding headers, each field is found by a **pattern on the question text**. The pipeline stops with a clear error if a required question is missing or matches more than one column, and logs any extra columns.

In [ ]:
FIELD_PATTERNS = {
    "timestamp":        r"^timestamp$",
    "email":            r"^email address$",
    "participant_code": r"^participant code",
    "session_date":     r"^date of the session",
    "finish_time":      r"^what time did you finish",
    "week":             r"^week of the plan",
    "session":          r"^session number",
    "session_type":     r"^session type",
    "status":           r"^did you complete the session",
    "modify_reason":    r"^briefly describe what you changed",
    "skip_reason":      r"^why did you skip",
    "skip_note":        r"^is there anything else you'd like to add",
    "rpe":              r"^how hard did your body feel",
    "q_pacing":         r"^did what you knew about your upcoming sessions",
    "tlx_mental":       r"^mental demand",
    "tlx_physical":     r"^physical demand",
    "tlx_temporal":     r"^temporal demand",
    "tlx_performance":  r"^performance",
    "tlx_effort":       r"^effort",
    "tlx_frustration":  r"^frustration",
    "q_tracking":       r"^what did you have to think about",
    "q_plan_mind":      r"^did anything on your mind about the plan",
    "q_other":          r"^is there anything else about today",
    "q_general":        r"^is there anything about this session in general",
}
# Google Forms exports a column for every question, even optional ones, so every field is required.
# Exception: the synthetic file was generated before the last Section 7 question was added.
OPTIONAL_FIELDS = {"q_general"} if IS_SYNTHETIC else set()
TLX_COLS = ["tlx_mental", "tlx_physical", "tlx_temporal", "tlx_performance", "tlx_effort", "tlx_frustration"]
RATING_COLS = ["rpe"] + TLX_COLS
FREE_TEXT_COLS = ["modify_reason", "skip_note", "q_pacing", "q_tracking", "q_plan_mind", "q_other", "q_general"]
TEXT_QS = {"q_pacing": "Did what you knew about upcoming sessions affect how hard you pushed?",
           "q_tracking": "What did you have to think about or keep track of?",
           "q_plan_mind": "Did anything on your mind about the plan make the run easier or harder?",
           "q_other": "Anything else that stood out?",
           "q_general": "Anything else about this session in general?"}

# ---------- The form's answer options (copied from the Google Form) ----------
RPE_LABELS = {6: "No exertion at all", 7: "Extremely light", 9: "Very light", 11: "Light", 13: "Somewhat hard",
              15: "Hard (heavy)", 17: "Very hard", 19: "Extremely hard", 20: "Maximal exertion"}
SKIP_REASONS = ["Illness", "Injury or soreness", "Schedule conflict", "Weather", "Low motivation"]   # + "Other:" (typed)
FORM_OPTIONS = {
    "participant_code": [f"R{i:02d}" for i in range(1, 11)],
    "week":             [f"Week {w}" for w in range(1, N_WEEKS + 1)],
    "session":          [f"Session {s}" for s in range(1, SESSIONS_PER_WEEK + 1)],
    "session_type":     ["Easy run", "Interval run", "Tempo run", "Long run"],
    "status":           ["As prescribed", "Modified", "Skipped"],
    "rpe":              [f"{v} - {RPE_LABELS[v]}" if v in RPE_LABELS else str(v) for v in range(6, 21)],
    **{c: [str(v) for v in range(0, 11)] for c in TLX_COLS},
}
STATUS_CODES = {"As prescribed": "prescribed", "Modified": "modified", "Skipped": "skipped"}
EXPERIENCE_GROUPS = ["Novice", "Experienced"]


class CleaningLog:
    """Collects one record per problem found, so every change to the data is traceable."""
    def __init__(self):
        self.records = []

    def add(self, step, issue, action, rows=None, detail=""):
        if rows is not None and len(rows) == 0:
            return                      # check ran, nothing found
        if rows is None:
            self.records.append(dict(step=step, participant=None, week=None, session=None,
                                     issue=issue, action=action, detail=detail))
            return
        def num(v):   # "Week 2" / "Session 3" (before parsing) and 2 / 3 (after) both become 2 / 3
            m = re.search(r"\d+", str(v)) if pd.notna(v) else None
            return int(m.group()) if m else None
        for _, r in rows.iterrows():
            self.records.append(dict(step=step, participant=r.get("participant_code"), week=num(r.get("week")),
                                     session=num(r.get("session")), issue=issue, action=action,
                                     detail=detail(r) if callable(detail) else detail))

    def frame(self):
        return pd.DataFrame(self.records, columns=["step", "participant", "week", "session", "issue", "action", "detail"])


def _norm_header(h):
    return re.sub(r"\s+", " ", str(h).replace("’", "'").replace("‘", "'")).strip().lower()


def match_columns(columns, log=None):
    """Map raw question headers to short field names. Raises on missing or ambiguous required fields."""
    rename, problems = {}, []
    for field, pattern in FIELD_PATTERNS.items():
        hits = [c for c in columns if re.search(pattern, _norm_header(c))]
        if len(hits) > 1:
            problems.append(f"'{field}' matches {len(hits)} columns: {hits}")
        elif not hits and field not in OPTIONAL_FIELDS:
            problems.append(f"required field '{field}' not found (pattern {pattern!r})")
        elif hits:
            rename[hits[0]] = field
    if problems:
        raise ValueError("Schema check failed:\n  " + "\n  ".join(problems))
    extra = [c for c in columns if c not in rename]
    if extra and log is not None:
        log.add("1 schema", "unexpected column(s)", "dropped", detail="; ".join(extra))
    return rename


def load_raw(path, log):
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    rename = match_columns(list(raw.columns), log)
    df = raw[list(rename)].rename(columns=rename)
    for f in sorted(OPTIONAL_FIELDS - set(df.columns)):
        df[f] = ""
        log.add("1 schema", "question not in this export", "added as a blank column", detail=f)
    df.insert(0, "row_id", np.arange(len(df)))
    return df


def check_key(key):
    """The participant key is typed by hand, so it must list exactly the form's codes, once each, with valid groups."""
    problems = []
    codes = set(FORM_OPTIONS["participant_code"])
    if key["participant_code"].duplicated().any():
        problems.append(f"duplicate codes: {sorted(key.loc[key['participant_code'].duplicated(), 'participant_code'])}")
    if set(key["participant_code"]) != codes:
        problems.append(f"codes missing from key: {sorted(codes - set(key['participant_code']))}; "
                        f"codes not on the form: {sorted(set(key['participant_code']) - codes)}")
    if bad := set(key["condition"]) - set(CONDITIONS):
        problems.append(f"unknown conditions: {sorted(bad)}")
    if bad := set(key["experience_group"]) - set(EXPERIENCE_GROUPS):
        problems.append(f"unknown experience groups: {sorted(bad)}")
    if problems:
        raise ValueError("Participant key problem:\n  " + "\n  ".join(problems))
    return key


def load_key(path):
    key = pd.read_csv(path, dtype=str, keep_default_na=False).apply(lambda s: s.str.strip())
    return check_key(key)


log = CleaningLog()
raw = load_raw(RAW_FILE, log)
key = load_key(KEY_FILE)
print(f"Raw responses: {len(raw)} rows × {raw.shape[1]} fields | Participants in key: {len(key)}")
print(key.groupby(["condition", "experience_group"]).size().unstack().to_string())
raw.head(3)

## 2. Data cleaning
The diary is a Google Form where the participant code, week, session number, session type, completion status, RPE, and all six TLX scales are **required choices**, the date and time are pickers, and the form branches on completion status. So typos, out-of-range ratings, and blank required answers cannot happen. Instead of quietly "fixing" them, **C2 stops the pipeline** if any answer is not one of the form's options. That can only mean the form or its export changed, and the settings need updating before any analysis is run.

The other checks target mistakes the form **cannot** prevent: picking the wrong option, submitting late or twice, or not submitting at all. Each check fixes or flags what it finds and records every action in the cleaning log.

| # | Problem the check catches | Action |
|---|---|---|
| C1 | Stray spaces and curly quotes; placeholder answers ("n/a", "-", ".") typed to get past a required text question | Tidied; placeholders **kept** and counted as a "No" (non-substantive) answer |
| C2 | Any answer that is not one of the form's options, a required answer left blank, or an answer to a question the branching would have hidden | **Pipeline stops** with a table of the rows (the form or export changed) |
| C3 | Corrections confirmed with the runner (`MANUAL_CORRECTIONS` in Section 0) | Applied and logged |
| C4 | Test entries submitted before the study started | Row dropped |
| C5 | Wrong participant code picked (the entry's email normally uses a different code) | Recoded to that email's usual code, or flagged if unclear; **email then deleted** (privacy, RA 10173) |
| C6 | Session type that contradicts the session number | Session number corrected only if the type **and** the date fit exactly one free slot; otherwise flagged |
| C7 | Session date outside the study, in a different week than reported, or after the submission | Flagged, kept (unless the runner confirmed the date: `CONFIRMED_DATES`) |
| C8 | Late entries (> 2 h after finishing), very late (> 24 h), submitted before finishing (often an AM/PM slip) | Flagged, kept; excluded in the sensitivity analysis |
| C9 | Duplicate submissions for the same session | Same date: one kept (policy setting), differences logged. Different dates: one kept and **flagged** (likely a wrong session number) |
| C10 | Sessions with no form at all | Added as `missing` rows so adherence is complete. Runners who stop logging stay `missing` and are flagged *stopped early*; only confirmed withdrawals become `withdrawn`; runners with no entries are `did_not_start` |
| C11 | Implausible answer patterns: identical TLX ratings on all six scales, RPE that contradicts physical demand | Flagged, kept |

Skip reasons typed under **Other:** are recorded as `Other`, with the typed text kept in `skip_reason_other`.

In [ ]:
PLACEHOLDER_PATTERN = r"n/?a|none|nil|null|-+|\.+"


class FormCheckError(ValueError):
    """An answer the current Google Form cannot produce: the form or its export has changed."""


def c1_tidy_text(df, log):
    df = df.copy()
    for col in df.columns.drop("row_id"):
        s = (df[col].astype("string")
             .str.replace("[‘’]", "'", regex=True).str.replace("[“”]", '"', regex=True)
             .str.replace(r"\s+", " ", regex=True).str.strip())
        df[col] = s.mask(s.eq(""))
    text = df[FREE_TEXT_COLS]
    placeholder = text.apply(lambda s: s.str.fullmatch(PLACEHOLDER_PATTERN, case=False)).fillna(False).astype(bool)
    log.add("C1 text", "placeholder answer ('n/a', '-', etc.)", "kept; counted as a 'No' (non-substantive) answer",
            df[placeholder.any(axis=1)], detail=lambda r: ", ".join(placeholder.columns[placeholder.loc[r.name]]))
    return df


def c2_check_form_values(df, log):
    """Stop if any answer is impossible on the current form; otherwise convert the answers to analysis types."""
    problems = []

    def bad(mask, field, problem):
        for _, r in df[mask].iterrows():
            problems.append(dict(row_id=r["row_id"], participant=r["participant_code"], field=field,
                                 value=r[field], problem=problem))

    status = df["status"]
    done, skipped = status.isin(["As prescribed", "Modified"]), status.eq("Skipped")
    for f in ["timestamp", "participant_code", "session_date", "finish_time", "week", "session",
              "session_type", "status"]:
        bad(df[f].isna(), f, "required answer is blank")
    for f in ["rpe", *TLX_COLS]:
        bad(done & df[f].isna(), f, "required answer is blank")
    for f in ["rpe", "q_pacing", *TLX_COLS, "q_tracking", "q_plan_mind"]:
        bad(skipped & df[f].notna(), f, "answered, but skipped sessions never see this question")
    # The three diary text questions turned out not to be required on the live form (R07 left some blank).
    # A blank answer is kept and treated like a placeholder (non-substantive), not a reason to stop.
    for f in ["q_pacing", "q_tracking", "q_plan_mind"]:
        log.add("C2 form", f"blank answer to {f}", "kept; counted as non-substantive (no answer)",
                df[done & df[f].isna()])
    bad(status.eq("Modified") & df["modify_reason"].isna(), "modify_reason", "required answer is blank")
    bad(skipped & df["skip_reason"].isna(), "skip_reason", "required answer is blank")
    for f in ["q_other", "q_general"]:
        bad(skipped & df[f].notna(), f, "answered, but skipped sessions never see this question")
    bad(~status.eq("Modified") & df["modify_reason"].notna(), "modify_reason", "only modified sessions see this question")
    for f in ["skip_reason", "skip_note"]:
        bad(~skipped & df[f].notna(), f, "only skipped sessions see this question")
    for f, options in FORM_OPTIONS.items():
        bad(df[f].notna() & ~df[f].isin(options), f, "not one of the form's options")
    parsed = {}
    for f, fmt, setting in (("timestamp", TIMESTAMP_FORMAT, "TIMESTAMP_FORMAT"),
                            ("session_date", DATE_FORMAT, "DATE_FORMAT"), ("finish_time", TIME_FORMAT, "TIME_FORMAT")):
        parsed[f] = pd.to_datetime(df[f], format=fmt, errors="coerce")
        bad(df[f].notna() & parsed[f].isna(), f, f"does not match {setting} ({fmt}) in Section 0")
    if problems:
        table = pd.DataFrame(problems)
        raise FormCheckError(
            f"{len(table)} answer(s) cannot come from the current Google Form, so the form or its export has changed. "
            "Update the settings (FORM_OPTIONS in Section 1, formats in Section 0) before running the analysis.\n"
            + table.head(25).to_string(index=False))

    df = df.copy()
    df["timestamp"] = parsed["timestamp"]
    df["session_date"] = parsed["session_date"].dt.normalize()
    df["finish_time"] = parsed["finish_time"] - parsed["finish_time"].dt.normalize()   # time of day as a Timedelta
    for col in ("week", "session"):
        df[col] = pd.to_numeric(df[col].str.extract(r"(\d+)")[0]).astype("Int64")
    df["status"] = df["status"].map(STATUS_CODES)
    df["rpe"] = pd.to_numeric(df["rpe"].str.extract(r"^(\d+)")[0])
    for col in TLX_COLS:
        df[col] = pd.to_numeric(df[col])
    if PERFORMANCE_REVERSED:
        df["tlx_performance"] = 10 - df["tlx_performance"]
    df[RATING_COLS] = df[RATING_COLS].astype("Float64")
    other = df["skip_reason"].notna() & ~df["skip_reason"].isin(SKIP_REASONS)
    df["skip_reason_other"] = df["skip_reason"].where(other)
    df.loc[other, "skip_reason"] = "Other"
    log.add("C2 form", "skip reason typed under 'Other'", "recorded as Other; text kept in skip_reason_other",
            df[other], detail=lambda r: f"'{r['skip_reason_other']}'")
    return df


def c3_manual_corrections(df, log):
    """Apply corrections the runner confirmed. Keys: (code as submitted, Google Forms timestamp)."""
    df = df.copy()
    allowed = {"participant_code", "week", "session", "drop"}
    for (code, stamp), change in MANUAL_CORRECTIONS.items():
        hit = df["participant_code"].eq(code) & df["timestamp"].eq(pd.to_datetime(stamp, format=TIMESTAMP_FORMAT))
        if hit.sum() != 1:
            raise ValueError(f"MANUAL_CORRECTIONS entry {(code, stamp)} matches {int(hit.sum())} rows (expected 1)")
        if bad := set(change) - allowed:
            raise ValueError(f"MANUAL_CORRECTIONS entry {(code, stamp)}: unknown keys {sorted(bad)}")
        if change.get("drop"):
            log.add("C3 confirmed", "entry confirmed as not a real diary entry", "row dropped", df[hit],
                    detail=str(change["drop"]))
            df = df[~hit]
            continue
        if change.get("participant_code", code) not in FORM_OPTIONS["participant_code"]:
            raise ValueError(f"MANUAL_CORRECTIONS entry {(code, stamp)}: unknown code {change['participant_code']}")
        if not 1 <= change.get("week", 1) <= N_WEEKS or not 1 <= change.get("session", 1) <= SESSIONS_PER_WEEK:
            raise ValueError(f"MANUAL_CORRECTIONS entry {(code, stamp)}: week or session outside the plan")
        log.add("C3 confirmed", "correction confirmed with the participant", "applied", df[hit],
                detail=", ".join(f"{k}: {df.loc[hit, k].iloc[0]} -> {v}" for k, v in change.items()))
        for k, v in change.items():
            df.loc[hit, k] = v
    return df


def c4_test_entries(df, log):
    pre = df["timestamp"] < STUDY_START - pd.Timedelta(days=1)
    log.add("C4 test entries", "submitted before the study started (test entry)", "row dropped", df[pre])
    return df[~pre].copy()


def c5_code_vs_email(df, log, min_rows=3, min_share=0.8):
    """A code picked by mistake shows up as an email submitting under a code that is not its usual one."""
    df = df.copy()
    df["flag_code_check"] = False
    email = df["email"].str.lower()
    for addr, grp in df.groupby(email):
        counts = grp["participant_code"].value_counts()
        if len(counts) == 1:
            continue
        if len(grp) >= min_rows and counts.iloc[0] / len(grp) >= min_share:
            main = counts.index[0]
            wrong = grp[grp["participant_code"] != main]
            log.add("C5 code check", "code does not match this email's usual code", f"recoded to {main}", wrong,
                    detail=lambda r: f"was {r['participant_code']}")
            df.loc[wrong.index, "participant_code"] = main
        else:
            log.add("C5 code check", "one email used several codes (no clear majority)", "kept, flagged; check with the runner",
                    grp, detail=", ".join(counts.index))
            df.loc[grp.index, "flag_code_check"] = True
    # Backup for emails with too few entries to recode: entries under a code that came from another email.
    email = df["email"].str.lower()
    for code, grp in df.groupby("participant_code"):
        counts = email[grp.index].value_counts()
        if len(counts) == 1:
            continue
        usual = counts.index[0] if counts.iloc[0] > counts.iloc[1] else None
        odd = grp if usual is None else grp[email[grp.index] != usual]
        log.add("C5 code check", "entry came from a different email than this code's usual one",
                "kept, flagged; check with the runner", odd, detail=f"{len(counts)} emails used this code")
        df.loc[odd.index, "flag_code_check"] = True
    return df.drop(columns="email")      # identifiers are removed once the check is done


def c6_session_labels(df, log):
    """Session type and session number are separate answers, so either can be the wrong click.
    The number is corrected only when its slot is already taken by another entry and the reported type and the date
    fit exactly one free slot that week. Otherwise the planned type is used and the row is flagged."""
    df = df.copy()
    df["flag_label_mismatch"] = False
    for idx in df.index[df["session_type"].ne(df["session"].map(lambda s: PLAN[s - 1]))]:
        r = df.loc[idx]
        week = df[df["participant_code"].eq(r["participant_code"]) & df["week"].eq(r["week"]) & (df.index != idx)]
        contested = week["session"].eq(r["session"]).any()
        fits = []
        for s in range(1, SESSIONS_PER_WEEK + 1):
            if PLAN[s - 1] != r["session_type"] or week["session"].eq(s).any():
                continue
            before, after = week.loc[week["session"] < s, "session_date"].max(), week.loc[week["session"] > s, "session_date"].min()
            if (pd.isna(before) or before <= r["session_date"]) and (pd.isna(after) or r["session_date"] <= after):
                fits.append(s)
        if contested and len(fits) == 1:
            log.add("C6 sessions", "session type does not match the session number", f"moved to Session {fits[0]}",
                    df.loc[[idx]], detail=f"answered '{r['session_type']}'; Session {r['session']} was already taken "
                                          f"and only Session {fits[0]} fits the type and date")
            df.loc[idx, "session"] = fits[0]
        else:
            log.add("C6 sessions", "session type does not match the session number",
                    "kept as reported, flagged; check with the runner", df.loc[[idx]],
                    detail=f"answered '{r['session_type']}', plan says '{PLAN[r['session'] - 1]}'")
            df.loc[idx, "flag_label_mismatch"] = True
    df["session_type"] = df["session"].map(lambda s: PLAN[s - 1])    # the plan defines the type
    return df


def c7_dates(df, log):
    df = df.copy()
    implied_week = (df["session_date"] - STUDY_START).dt.days // 7 + 1
    outside = ~implied_week.between(1, N_WEEKS)
    mismatch = ~outside & implied_week.ne(df["week"])
    confirmed = pd.Series([(c, w, s) in CONFIRMED_DATES for c, w, s in
                           zip(df["participant_code"], df["week"], df["session"])], index=df.index)
    log.add("C7 dates", "session date in a different week than reported, confirmed with the runner",
            "kept, not flagged (run late)", df[mismatch & confirmed])
    mismatch = mismatch & ~confirmed
    future = df["session_date"] > df["timestamp"].dt.normalize()
    df["flag_date_problem"] = outside | mismatch | future
    log.add("C7 dates", "session date outside the study period", "flagged (kept)", df[outside])
    log.add("C7 dates", "session date falls in a different week than reported", "flagged (kept)", df[mismatch])
    log.add("C7 dates", "session date is after the submission date", "flagged (kept)", df[future])
    return df


def c8_timing(df, log):
    df = df.copy()
    done = df["status"].ne("skipped")
    finish_dt = df["session_date"] + df["finish_time"]
    lag = (df["timestamp"] - finish_dt).dt.total_seconds() / 3600
    negative = done & (lag < 0)
    log.add("C8 timing", "submitted before the reported finish time", "lag set to missing, flagged", df[negative],
            detail=lambda r: f"{lag[r.name]:.2f} h")
    df["finish_dt"] = finish_dt.where(done)
    df["submit_lag_h"] = lag.where(done & ~negative)
    df["flag_negative_lag"] = negative
    df["flag_late"] = df["submit_lag_h"] > LATE_THRESHOLD_H
    df["flag_very_late"] = df["submit_lag_h"] > VERY_LATE_H
    log.add("C8 timing", f"submitted more than {LATE_THRESHOLD_H:g} h after finishing", "flagged (kept)",
            df[df["flag_late"]], detail=lambda r: f"{r['submit_lag_h']:.1f} h")
    return df


def c9_duplicates(df, log, policy=None):
    policy = policy or DUPLICATE_POLICY          # read at call time so the setting can be changed
    df = df.sort_values("timestamp").copy()
    k = ["participant_code", "week", "session"]
    dup = df.duplicated(k, keep=False)
    df["flag_duplicate_resolved"] = False
    keep_idx = []
    for _, grp in df[dup].groupby(k):
        kept = grp.iloc[-1] if policy == "last" else grp.iloc[0]
        keep_idx.append(kept.name)
        same_day = grp["session_date"].nunique() == 1
        if not same_day:
            df.loc[kept.name, "flag_label_mismatch"] = True
        compare = [c for c in df.columns if c not in ("row_id", "timestamp", "finish_dt", "submit_lag_h")]
        for _, other in grp.drop(index=kept.name).iterrows():
            diff = [f"{c}: {other[c]} -> {kept[c]}" for c in compare
                    if not (pd.isna(other[c]) and pd.isna(kept[c])) and str(other[c]) != str(kept[c])]
            if same_day:
                issue, action = f"duplicate submission ({len(grp)} entries)", f"kept the {policy} entry"
            else:
                issue = "two entries for one session on different dates (one may have the wrong session number)"
                action = f"kept the {policy} entry, flagged; check with the runner"
            log.add("C9 duplicates", issue, action, grp.loc[[other.name]], detail="; ".join(diff) or "identical answers")
    drop = df[dup].index.difference(keep_idx)
    df.loc[keep_idx, "flag_duplicate_resolved"] = True
    return df.drop(index=drop)


def c10_expected_grid(df, key, log):
    codes = sorted(set(key["participant_code"]) - DATA_WITHDRAWN)
    grid = pd.DataFrame(list(itertools.product(codes, range(1, N_WEEKS + 1), range(1, SESSIONS_PER_WEEK + 1))),
                        columns=["participant_code", "week", "session"])
    grid[["week", "session"]] = grid[["week", "session"]].astype("Int64")
    out = grid.merge(df[df["participant_code"].isin(codes)], on=["participant_code", "week", "session"], how="left")
    out["session_index"] = (out["week"] - 1) * SESSIONS_PER_WEEK + out["session"]
    out["session_type"] = out["session"].map(lambda s: PLAN[s - 1])
    no_form = out["status"].isna()
    out.loc[no_form, "status"] = "missing"
    for code, grp in out.sort_values("session_index").groupby("participant_code"):
        mine = out["participant_code"] == code
        if grp["status"].eq("missing").all():
            out.loc[mine, "status"] = "did_not_start"
            log.add("C10 grid", "participant enrolled but never submitted a diary entry",
                    "all sessions marked did_not_start (left out of adherence denominators)", out[mine].head(1))
            continue
        if code in CONFIRMED_WITHDRAWALS:
            last_ok = CONFIRMED_WITHDRAWALS[code]
            w = mine & (out["session_index"] > last_ok) & out["status"].eq("missing")
            out.loc[w, "status"] = "withdrawn"
            log.add("C10 grid", "participant confirmed a withdrawal", "remaining sessions marked withdrawn",
                    out[w].head(1), detail=f"withdrew after session index {last_ok}; {int(w.sum())} sessions not expected")
            continue
        trailing = 0
        for st in grp["status"].iloc[::-1]:
            if st != "missing":
                break
            trailing += 1
        if trailing >= STOPPED_EARLY_RUN:
            log.add("C10 grid", "participant stopped logging before the end of the plan",
                    "kept as missing (not a withdrawal); flagged as stopped early", grp.head(1),
                    detail=f"last logged session index {len(grp) - trailing}; {trailing} unlogged sessions at the end")
    log.add("C10 grid", "scheduled session with no form submitted", "added as 'missing'",
            out[out["status"].eq("missing")])
    for c in [c for c in out.columns if c.startswith("flag_")]:
        out[c] = out[c].fillna(False).astype(bool)
    if DATA_WITHDRAWN:
        log.add("C10 grid", "participant asked for data removal", "all rows dropped",
                detail=", ".join(sorted(DATA_WITHDRAWN)))
    return out


def c11_plausibility(df, log):
    df = df.copy()
    t = df[TLX_COLS]
    df["flag_straightline"] = t.notna().all(axis=1) & t.nunique(axis=1).eq(1)
    log.add("C11 plausibility", "identical rating on all six TLX scales", "flagged (kept)", df[df["flag_straightline"]])
    hi = (df["rpe"] >= 15) & (df["tlx_physical"] <= 2)
    lo = (df["rpe"] <= 8) & (df["tlx_physical"] >= 8)
    df["flag_rpe_mismatch"] = (hi | lo).fillna(False).astype(bool)
    log.add("C11 plausibility", "RPE contradicts physical demand", "flagged (kept)", df[df["flag_rpe_mismatch"]],
            detail=lambda r: f"RPE {r['rpe']}, physical {r['tlx_physical']}")
    return df


def clean(raw, key, log):
    df = c1_tidy_text(raw, log)
    df = c2_check_form_values(df, log)
    df = c3_manual_corrections(df, log)
    df = c4_test_entries(df, log)
    df = c5_code_vs_email(df, log)
    df = c6_session_labels(df, log)
    df = c7_dates(df, log)
    df = c8_timing(df, log)
    df = c9_duplicates(df, log)
    df = c10_expected_grid(df, key, log)
    df = c11_plausibility(df, log)
    return df.sort_values(["participant_code", "session_index"]).reset_index(drop=True)


clean_df = clean(raw, key, log)
log_df = log.frame()
log_df.to_csv(OUT_DIR / "cleaning_log.csv", index=False)

summary = (log_df[log_df["participant"].notna() | log_df["detail"].ne("")]
           .groupby(["step", "issue", "action"], sort=False).size().rename("rows").reset_index())
print(f"{len(raw)} raw rows -> {int(clean_df['status'].isin(['prescribed','modified','skipped']).sum())} valid entries "
      f"+ {int(clean_df['status'].eq('missing').sum())} missing + {int(clean_df['status'].eq('withdrawn').sum())} withdrawn "
      f"+ {int(clean_df['status'].eq('did_not_start').sum())} did not start "
      f"= {len(clean_df)} scheduled sessions")
summary

**Cleaning log detail.** Every row-level action, so any change can be traced back to a specific participant and session.

In [ ]:
log_df[log_df["participant"].notna()].reset_index(drop=True)

## 3. Data preparation
The cleaned rows are turned into **analysis-ready tables**. Each field has one meaning, one type, and one unit:

* `sessions`: one row per participant × scheduled session (including missing, withdrawn, and did-not-start sessions), with condition, experience, plan position, ratings on their analysis scale (TLX 0–100), Raw TLX, quality flags, and adjusted scores for the anticipatory pacing analysis.
* `tlx_long`: TLX in long format (one row per subscale), for subscale charts.
* `participant_summary`: one row per runner (the unit for between-condition comparisons, because sessions from the same runner are not independent).
* `text_responses`: one row per open-ended answer, kept exactly as written, with a `substantive` flag for answers that say more than "No" (English or Filipino, e.g. "Wala naman") and a `language` tag (`English` / `Filipino or Taglish`). Runners may answer in either language; the tag lets coders route Filipino and Taglish answers to a bilingual human or LLM coder, since keyword patterns are English-only, and lets the analysis check whether language use differs by condition.

**Raw TLX** is the unweighted mean of the six subscales (Hart, 2006), computed only when all six are present. Each subscale is multiplied by 10 to put it on the standard 0–100 scale.

In [ ]:
TERSE_RE = re.compile(r"^(no|nope|none|nothing|not really|neither|not today|nothing in particular|n/?a|"
                      r"no,? just ran it|nothing much|not sure|idk|same|nil|null|-+|\.+|"
                      # Filipino / Taglish equivalents of "No" / "Nothing" / "Not really" / "It was fine"
                      r"(wala|hindi|di|hinde)( naman| po| masyado| talaga| pa| rin| din)*|"
                      r"(okay|ok|oks|ayos|goods) lang( naman)?( po)?|wala akong (ma)?isip|same lang|ganun pa rin)"
                      r"[.!]?$", re.I)

# Common Filipino words that do not occur as English words, used to tag answers that English keyword patterns cannot read.
FILIPINO_MARKERS = {
    "ang", "ng", "mga", "sa", "ko", "ako", "akong", "kong", "mo", "ka", "siya", "kami", "tayo", "namin", "natin",
    "yung", "iyong", "ung", "yun", "iyon", "naman", "kasi", "lang", "din", "rin", "po", "hindi", "di", "wala",
    "meron", "oo", "sobra", "medyo", "talaga", "parang", "kaya", "pero", "tapos", "nung", "noong", "para", "ito",
    "yan", "dahil", "alam", "bukas", "ngayon", "kanina", "mas", "lahat", "isang", "nang", "kung", "pag", "nag",
    "masyado", "konti", "nako", "grabe", "kahit", "pala", "sana", "nalang", "na", "pa", "ba", "eh",
}


def is_substantive(text):
    return isinstance(text, str) and len(text.strip()) > 0 and not TERSE_RE.match(text.strip())


def detect_language(text):
    """'Filipino or Taglish' if the answer contains any common Filipino word (including prefixed forms such as
    'nag-hold' or 'mag-pace'), else 'English'. Code-switching is normal, so the two are not separated further."""
    words = re.findall(r"[a-zñ]+(?:-[a-zñ]+)*", str(text).lower())
    tokens = set(words) | {w.split("-")[0] for w in words if "-" in w}
    return "Filipino or Taglish" if tokens & (FILIPINO_MARKERS | {"mag", "nag", "pag", "ma", "na"}) else "English"


def next_session_info(week, session):
    if session < SESSIONS_PER_WEEK:
        nxt = PLAN[session]
    elif week < N_WEEKS:
        nxt = PLAN[0]
    else:
        return None, "final session"
    return nxt, ("harder session next" if nxt in HARD_TYPES else "easy run next")


def additive_residual(df, y, factors=("participant_code", "session_type")):
    """Residual of y after removing each runner's typical level and each session type's typical level (OLS)."""
    d = df[[y, *factors]].dropna()
    X = pd.get_dummies(d[list(factors)].astype(str), drop_first=True, dtype=float)
    X.insert(0, "const", 1.0)
    beta, *_ = np.linalg.lstsq(X.to_numpy(), d[y].to_numpy(dtype=float), rcond=None)
    out = pd.Series(np.nan, index=df.index)
    out[d.index] = d[y].to_numpy(dtype=float) - X.to_numpy() @ beta
    return out


STATUS_ORDER = ["prescribed", "modified", "skipped", "missing", "withdrawn", "did_not_start"]
NOT_EXPECTED = ["withdrawn", "did_not_start"]   # sessions left out of adherence denominators


def trailing_missing(status):
    """Number of 'missing' sessions in a row at the end of one runner's plan."""
    n = 0
    for st in list(status)[::-1]:
        if st != "missing":
            break
        n += 1
    return n


def prepare(clean_df, key):
    s = clean_df.merge(key, on="participant_code", how="left", validate="many_to_one")
    s["condition"] = pd.Categorical(s["condition"], CONDITIONS)
    s["experience_group"] = pd.Categorical(s["experience_group"], ["Novice", "Experienced"])
    s["status"] = pd.Categorical(s["status"], STATUS_ORDER)
    s["session_type"] = pd.Categorical(s["session_type"], ["Easy run", "Interval run", "Tempo run", "Long run"])
    s["submitted"] = s["status"].isin(["prescribed", "modified", "skipped"])
    s["completed"] = s["status"].isin(["prescribed", "modified"])
    s["day_of_week"] = s["session_date"].dt.day_name()
    info = [next_session_info(w, n) for w, n in zip(s["week"], s["session"])]
    s["next_session_type"] = [i[0] for i in info]
    s["next_category"] = pd.Categorical([i[1] for i in info],
                                        ["harder session next", "easy run next", "final session"])
    for c in TLX_COLS:
        s[f"{c}_100"] = (s[c] * 10).astype("Float64")
    s["tlx_raw"] = s[[f"{c}_100" for c in TLX_COLS]].astype(float).mean(axis=1, skipna=False)
    s["rpe"] = s["rpe"].astype(float)
    s["mental"] = s["tlx_mental_100"].astype(float)
    s["rpe_adj"] = additive_residual(s, "rpe")
    s["mental_adj"] = additive_residual(s, "mental")
    s["any_quality_flag"] = s[["flag_late", "flag_negative_lag", "flag_date_problem", "flag_code_check",
                               "flag_label_mismatch", "flag_straightline", "flag_rpe_mismatch"]].any(axis=1)
    s["skip_reason"] = pd.Categorical(s["skip_reason"], SKIP_REASONS + ["Other"])
    cols = ["participant_code", "condition", "experience_group", "week", "session", "session_index",
            "session_type", "next_session_type", "next_category", "session_date", "day_of_week", "finish_dt",
            "timestamp", "submit_lag_h", "status", "submitted", "completed", "skip_reason", "skip_reason_other",
            "rpe", "mental", *[f"{c}_100" for c in TLX_COLS], "tlx_raw", "rpe_adj", "mental_adj",
            "flag_late", "flag_very_late", "flag_negative_lag", "flag_date_problem", "flag_code_check",
            "flag_label_mismatch", "flag_duplicate_resolved", "flag_straightline", "flag_rpe_mismatch",
            "any_quality_flag"]
    sessions = s[cols].rename(columns={"timestamp": "submitted_at"})
    text = (s.melt(id_vars=["participant_code", "condition", "week", "session", "session_index", "session_type",
                            "next_category", "status"],
                   value_vars=["q_pacing", "q_tracking", "q_plan_mind", "q_other", "q_general", "modify_reason"],
                   var_name="question", value_name="response")
            .dropna(subset=["response"]))
    text["substantive"] = text["response"].map(is_substantive)
    text["language"] = pd.Categorical(text["response"].map(detect_language), ["English", "Filipino or Taglish"])
    text = text.sort_values(["participant_code", "session_index", "question"]).reset_index(drop=True)
    text.insert(0, "response_id", [f"T{i:04d}" for i in range(len(text))])
    return sessions, text


def tlx_to_long(sessions):
    labels = {f"{c}_100": c.replace("tlx_", "").title() for c in TLX_COLS}
    long = sessions[sessions["completed"]].melt(
        id_vars=["participant_code", "condition", "experience_group", "week", "session_index", "session_type"],
        value_vars=list(labels), var_name="subscale", value_name="score").dropna(subset=["score"])
    long["subscale"] = pd.Categorical(long["subscale"].map(labels), list(labels.values()))
    return long


def summarize_participants(sessions):
    g = sessions.groupby("participant_code", observed=True)
    done = sessions[sessions["completed"]].groupby("participant_code", observed=True)
    expected = sessions[~sessions["status"].isin(NOT_EXPECTED)].groupby("participant_code", observed=True).size()
    ordered = sessions.sort_values("session_index").groupby("participant_code", observed=True)["status"]
    out = pd.DataFrame({
        "condition": g["condition"].first(), "experience_group": g["experience_group"].first(),
        "sessions_expected": expected,
        "submitted": g["submitted"].sum(), "prescribed": g["status"].apply(lambda x: (x == "prescribed").sum()),
        "modified": g["status"].apply(lambda x: (x == "modified").sum()),
        "skipped": g["status"].apply(lambda x: (x == "skipped").sum()),
        "missing": g["status"].apply(lambda x: (x == "missing").sum()),
        "withdrawn": g["status"].apply(lambda x: (x == "withdrawn").sum()) > 0,
        "did_not_start": g["status"].apply(lambda x: (x == "did_not_start").all()),
        "unlogged_at_end": ordered.apply(trailing_missing),
        "late_entries": g["flag_late"].sum(),
    })
    out["submission_rate"] = out["submitted"] / out["sessions_expected"]
    out["completion_rate"] = (out["prescribed"] + out["modified"]) / out["sessions_expected"]
    out["stopped_early"] = out["unlogged_at_end"] >= STOPPED_EARLY_RUN
    out["below_min_submission"] = out["submission_rate"] < MIN_SUBMISSION_RATE   # proposal's 50% rule (not applied)
    for col in ["rpe", "mental", *[f"{c}_100" for c in TLX_COLS[1:]], "tlx_raw"]:
        out[f"mean_{col}"] = done[col].mean()
    out["sd_rpe"] = done["rpe"].std()
    out["sd_mental"] = done["mental"].std()
    return out.reset_index()


sessions, text_responses = prepare(clean_df, key)
tlx_long = tlx_to_long(sessions)
participants = summarize_participants(sessions)

DATA_DICTIONARY = {
    "participant_code": "Participant ID (R01-R10); no names or emails are kept",
    "condition": "Full Plan or Current Only (from the participant key)",
    "experience_group": "Novice (<1 year running) or Experienced (>=1 year); stratification variable",
    "week": "Week of the plan (1..N_WEEKS)", "session": "Session number within the week (1-5)",
    "session_index": "Position in the whole plan (1..N_WEEKS*5)",
    "session_type": "Planned session type (from the plan, not the participant's answer)",
    "next_session_type": "Type of the following session (blank after the final session)",
    "next_category": "harder session next / easy run next / final session (for anticipatory pacing)",
    "session_date": "Date the participant ran (or skipped)", "day_of_week": "Weekday of session_date",
    "finish_dt": "Date and time the run finished", "submitted_at": "Google Forms timestamp",
    "submit_lag_h": "Hours from finishing to submitting (completed sessions only)",
    "status": "prescribed / modified / skipped / missing (no form) / withdrawn (confirmed withdrawal only) / did_not_start (enrolled, never logged)",
    "submitted": "A form was received for this session", "completed": "Ran as prescribed or modified",
    "skip_reason": "Reason given for skipping (one of the form's options, or Other)",
    "skip_reason_other": "Text typed under 'Other:' as the skip reason",
    "rpe": "Borg RPE, 6-20", "mental": "NASA-TLX mental demand, 0-100 (same as tlx_mental_100)",
    "tlx_mental_100": "Mental demand 0-100", "tlx_physical_100": "Physical demand 0-100",
    "tlx_temporal_100": "Temporal demand 0-100", "tlx_performance_100": "Performance 0-100 (0 = perfect, 100 = failure)",
    "tlx_effort_100": "Effort 0-100", "tlx_frustration_100": "Frustration 0-100",
    "tlx_raw": "Raw TLX: mean of the six subscales, 0-100 (blank if any subscale is missing)",
    "rpe_adj": "RPE minus the runner's usual level and the session type's usual level (OLS residual)",
    "mental_adj": "Mental demand adjusted the same way as rpe_adj",
    "flag_late": f"Submitted more than {LATE_THRESHOLD_H:g} h after finishing",
    "flag_very_late": f"Submitted more than {VERY_LATE_H:g} h after finishing",
    "flag_negative_lag": "Reported finish time is after the submission time",
    "flag_date_problem": "Session date outside the study, in the wrong week, or after submission",
    "flag_code_check": "Participant code may be wrong (the email did not match the code's usual one); check with the runner",
    "flag_label_mismatch": "Session number may be wrong (type contradicts it, or two entries on different dates); check with the runner",
    "flag_duplicate_resolved": "This row is the one kept from a duplicate submission",
    "flag_straightline": "Same value on all six TLX scales", "flag_rpe_mismatch": "RPE contradicts physical demand",
    "any_quality_flag": "Any of the data-quality flags above (excluding duplicate_resolved)",
}
dictionary = pd.DataFrame({
    "field": sessions.columns,
    "type": [str(t) for t in sessions.dtypes],
    "description": [DATA_DICTIONARY.get(c, "") for c in sessions.columns],
    "% blank": (sessions.isna().mean() * 100).round(1).values,
    "example": [sessions[c].dropna().iloc[0] if sessions[c].notna().any() else "" for c in sessions.columns],
})
assert dictionary["description"].ne("").all(), "every field needs a description"

sessions.to_csv(OUT_DIR / "sessions.csv", index=False)
tlx_long.to_csv(OUT_DIR / "tlx_long.csv", index=False)
participants.to_csv(OUT_DIR / "participant_summary.csv", index=False)
text_responses.to_csv(OUT_DIR / "text_responses.csv", index=False)
dictionary.to_csv(OUT_DIR / "data_dictionary.csv", index=False)
print(f"sessions: {sessions.shape} | tlx_long: {tlx_long.shape} | participants: {participants.shape} | "
      f"text_responses: {text_responses.shape}")
print("Answers by language and condition:")
print(pd.crosstab(text_responses["condition"], text_responses["language"]).to_string())
dictionary

In [ ]:
sessions[sessions["completed"]].head(8)

In [ ]:
participants.round(2)

## 4. Analysis toolkit
All statistical and coding functions are defined once here so they can be tested (Section 5) before they are used.

**Unit of analysis.** Sessions from the same runner are not independent, so condition comparisons use **one mean per runner** (n = 6 vs 6). Session-level data are used only for within-runner questions (trajectories, anticipatory pacing, RQ3 within-person correlation).

**Statistics, in line with the proposal.** The primary analysis is descriptive (mean, SD, median, range per condition). Because the sample is small, any inferential numbers are **exploratory** and are reported as effect sizes with confidence intervals:

| Tool | Why this one |
|---|---|
| Hedges' *g* + bootstrap 95% CI | Standardized difference with small-sample correction; CI shows the uncertainty honestly |
| **Stratified exact permutation test** | Re-runs every possible random assignment that our stratified design could have produced (3 novices + 3 experienced per condition = 400 assignments). No normality assumption, exact for n = 12 |
| Holm correction | Controls false positives across the outcomes compared |
| Linear mixed model (random intercept per runner) | Uses every session while accounting for repeated measures; adjusts for session type, week, and experience |
| Repeated-measures correlation (Bakdash & Marusich, 2017) | RQ3: does mental demand rise and fall with RPE *within* a runner, without the between-runner differences that a plain Spearman mixes in |

In [ ]:
OUTCOMES = {"mental": "Mental demand (0-100)", "rpe": "Borg RPE (6-20)", "tlx_raw": "Raw TLX (0-100)"}


def participant_means(sessions, cols, mask=None):
    keep = sessions["completed"] if mask is None else sessions["completed"] & mask
    return (sessions[keep].groupby(["participant_code", "condition", "experience_group"], observed=True)[cols]
            .mean().reset_index())


def describe_by_condition(pm, cols, labels=None):
    rows = []
    for col in cols:
        for cond in CONDITIONS:
            v = pm.loc[pm["condition"] == cond, col].dropna()
            rows.append({"measure": (labels or {}).get(col, col), "condition": cond, "n runners": len(v),
                         "mean": v.mean(), "SD": v.std(), "median": v.median(), "min": v.min(), "max": v.max()})
    return pd.DataFrame(rows).set_index(["measure", "condition"])


def hedges_g(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    n1, n2 = len(a), len(b)
    sp = np.sqrt(((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1)) / (n1 + n2 - 2))
    return (a.mean() - b.mean()) / sp * (1 - 3 / (4 * (n1 + n2) - 9))


def bootstrap_diff(pm, col, n_boot=5000, seed=SEED):
    """Bootstrap the Full Plan minus Current Only difference in means, resampling runners within condition."""
    g = np.random.default_rng(seed)
    a = pm.loc[pm["condition"] == "Full Plan", col].dropna().to_numpy(float)
    b = pm.loc[pm["condition"] == "Current Only", col].dropna().to_numpy(float)
    diffs = np.array([g.choice(a, len(a)).mean() - g.choice(b, len(b)).mean() for _ in range(n_boot)])
    gs = np.array([hedges_g(g.choice(a, len(a)), g.choice(b, len(b))) for _ in range(n_boot)])
    gs = gs[np.isfinite(gs)]
    return dict(diff=a.mean() - b.mean(), diff_lo=np.percentile(diffs, 2.5), diff_hi=np.percentile(diffs, 97.5),
                g=hedges_g(a, b), g_lo=np.percentile(gs, 2.5), g_hi=np.percentile(gs, 97.5), boot=diffs)


def stratified_permutation_test(pm, col, strata="experience_group", max_exact=200_000, seed=SEED):
    """Two-sided test of the difference in means, permuting condition labels within each stratum."""
    d = pm[["participant_code", "condition", strata, col]].dropna()
    values = d.set_index("participant_code")[col]
    obs = values[d["condition"].eq("Full Plan").to_numpy()].mean() - values[d["condition"].eq("Current Only").to_numpy()].mean()
    per_stratum = []
    for _, grp in d.groupby(strata, observed=True):
        ids = list(grp["participant_code"])
        k = int((grp["condition"] == "Full Plan").sum())
        per_stratum.append([set(c) for c in itertools.combinations(ids, k)])
    n_total = int(np.prod([len(p) for p in per_stratum]))
    all_ids = set(values.index)
    if n_total <= max_exact:
        assignments = (set().union(*combo) for combo in itertools.product(*per_stratum))
    else:
        g = np.random.default_rng(seed)
        assignments = (set().union(*[p[g.integers(len(p))] for p in per_stratum]) for _ in range(20_000))
    stats_ = np.array([values[list(fp)].mean() - values[list(all_ids - fp)].mean() for fp in assignments])
    p = np.mean(np.abs(stats_) >= abs(obs) - 1e-12)
    return dict(diff=obs, p=p, n_assignments=len(stats_), exact=n_total <= max_exact)


def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj


def fit_mixed(sessions, y):
    d = sessions[sessions["completed"]].dropna(subset=[y]).copy()
    d["week"] = d["week"].astype(float)
    for c in ["condition", "session_type", "experience_group"]:
        d[c] = d[c].astype(str)
    # Same specification as the Section 8 primary model. Week and experience are left out: with one Experienced
    # runner with data, the experience term is essentially that runner's intercept.
    formula = f"{y} ~ C(condition, Treatment('Current Only')) + C(session_type, Treatment('Easy run'))"
    d[y] = d[y].astype(float)
    res, errors = None, []
    for method in ("lbfgs", "bfgs", "powell", "nm"):     # fall back if an optimizer fails to converge
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                res = smf.mixedlm(formula, d, groups=d["participant_code"]).fit(reml=True, method=method)
            if np.all(np.isfinite(res.bse)):
                break
        except (np.linalg.LinAlgError, ValueError) as e:
            errors.append(f"{method}: {e}")
            res = None
    if res is None:
        raise RuntimeError(f"mixed model for {y} did not converge: {errors}")
    name = [n for n in res.params.index if n.startswith("C(condition")][0]
    ci = res.conf_int().loc[name]
    return dict(outcome=y, estimate=res.params[name], ci_lo=ci.iloc[0], ci_hi=ci.iloc[1], p=res.pvalues[name], optimizer=method,
                n_sessions=len(d), n_runners=d["participant_code"].nunique(),
                runner_sd=float(np.sqrt(res.cov_re.iloc[0, 0])), residual_sd=float(np.sqrt(res.scale)), model=res)


def rmcorr(df, x, y, subject="participant_code"):
    """Repeated-measures correlation (Bakdash & Marusich, 2017): common within-subject association."""
    d = df[[x, y, subject]].dropna()
    d = d[d.groupby(subject)[x].transform("size") >= 2]
    xc = d[x] - d.groupby(subject)[x].transform("mean")
    yc = d[y] - d.groupby(subject)[y].transform("mean")
    sxx, syy, sxy = (xc ** 2).sum(), (yc ** 2).sum(), (xc * yc).sum()
    r = sxy / np.sqrt(sxx * syy)
    dof = len(d) - d[subject].nunique() - 1
    t = r * np.sqrt(dof / (1 - r ** 2))
    return dict(r=r, dof=dof, p=2 * stats.t.sf(abs(t), dof), n_obs=len(d), n_subjects=d[subject].nunique())


def rmcorr_boot_ci(df, x, y, subject="participant_code", n_boot=2000, seed=SEED):
    """Cluster bootstrap: resample whole runners, so the CI respects the repeated measures."""
    g = np.random.default_rng(seed)
    subjects = df[subject].dropna().unique()
    groups = {s: df[df[subject] == s] for s in subjects}
    rs = []
    for _ in range(n_boot):
        pick = g.choice(subjects, len(subjects))
        boot = pd.concat([groups[s].assign(_id=i) for i, s in enumerate(pick)])
        r = rmcorr(boot, x, y, "_id")["r"]
        if np.isfinite(r):
            rs.append(r)
    return np.percentile(rs, [2.5, 97.5])


def per_runner_spearman(df, x, y, subject="participant_code"):
    rows = []
    for sid, grp in df.dropna(subset=[x, y]).groupby(subject):
        n = len(grp)
        rho = stats.spearmanr(grp[x], grp[y]).statistic if grp[x].nunique() > 1 and grp[y].nunique() > 1 else np.nan
        se = np.sqrt(1.06 / (n - 3)) if n > 3 else np.nan          # Fieller et al. (1957) SE for Spearman
        z = np.arctanh(np.clip(rho, -0.999, 0.999))
        rows.append(dict(participant_code=sid, condition=grp["condition"].iloc[0], n=n, rho=rho,
                         lo=np.tanh(z - 1.96 * se), hi=np.tanh(z + 1.96 * se)))
    return pd.DataFrame(rows)


def cohen_kappa(a, b):
    a, b = pd.Series(a).astype(str).to_numpy(), pd.Series(b).astype(str).to_numpy()
    labels = np.union1d(a, b)
    po = np.mean(a == b)
    pe = sum(np.mean(a == l) * np.mean(b == l) for l in labels)
    return 1.0 if pe == 1 else (po - pe) / (1 - pe)

### 4b. Content analysis: codebook, rule-based coder, and LLM coder
The open-ended diary answers are coded with a **deductive codebook** derived from the conceptual framework (anticipatory regulation, cognitive load, goal progress). Two independent coders apply it:

1. **Rule-based coder**: transparent keyword patterns. Fully reproducible, but literal.
2. **LLM coder**: an LLM applies the same written definitions. It handles paraphrase better, but must be checked.

**Safeguards for the LLM coder:** it is **blind to condition and participant** (it only sees an anonymous ID and the text, so it cannot code toward our expectations), it must return strict JSON that is validated against the codebook, every prompt and raw reply is saved in `processed/llm_coding/` as an audit trail, and its codes are compared with the rule-based coder (Cohen's κ) and with a human-coded sample before they are used.

Set `LLM_PROVIDER` to `"anthropic"` or `"openai"` (with an API key in the environment) to call an API, or leave it as `"manual"`: the notebook then writes prompt files you can paste into any chat LLM, and reads back the replies you save next to them. Until LLM codes exist, the analysis uses the rule-based codes.

**Decision for this study:** the LLM coder is **not used** (`LLM_PROVIDER = "manual"` and no replies are pasted back), because sending diary text to a third-party API is not covered by the consent form. The rule-based codebook below was revised after reading the real answers (themes such as outside demands, session framing, and preparing), and it is validated against a **human-coded 25% sample** (Cohen's κ, Section 10b).

In [ ]:
LLM_PROVIDER = "manual"           # "manual", "anthropic", or "openai"
LLM_MODEL = os.environ.get("LLM_MODEL", "")   # e.g. a current Claude or GPT model ID when using an API
LLM_BATCH_SIZE = 40

CODEBOOK = {
    # Revised on the real answers. Patterns are generic phrasings, never participant quotes.
    # Known misfires of the first version are fixed: tiredness from the run itself is not an outside demand,
    # "recovery periods" is not a health factor, "manage my time" to fit the run in is not an outside demand,
    # and anticipating *unknown* upcoming sessions is uncertainty, not looking ahead.
    "q_pacing": {
        "single_name": "pacing",
        "single": {   # exactly one per answer; checked in this order
            "PREPARED": ("Used what they knew to prepare mentally or physically, or to set the effort level",
                         r"prepar|mental idea of how much|the level (and the|of) effort|took a (short )?nap"),
            "HELD_BACK": ("Held back, took it easy, or saved energy",
                          r"held back|hold back|in reserve|eased off|conserv|kept it light|just relaxed|brisk walk|take it easy|taking it slow"
                          r"|(didn't|did not|without|avoid|not) push(ing)? myself too hard|not to use up|manage my effort|rather than running faster|cautious"),
            "PUSHED": ("Pushed harder than usual", r"pushed|worked harder|went all out|gave it everything|motivated to finish|run a (little )?(bit )?fast"),
            "UNSURE": ("Unsure / went by feel", r"hard to say|went by feel|not sure|\bidk\b"),
            "NO_EFFECT": ("No effect on effort", r"^(no|not really|not at all|not today|nope)\b|didn't affect|did not affect"),
        },
        "multi": {
            "KNEW_UPCOMING": ("Refers to a known upcoming session or the end of the plan",
                              r"future sessions|following sessions|affect the next|last (run|session)|was next|is next|next one is|is coming up"),
            "UNCERTAIN_UPCOMING": ("Mentions not knowing what comes next",
                                   r"unsure of what to expect|(didn't|did not|don't) know what|not knowing|no idea what"),
            "TODAY_SESSION": ("Bases effort on today's session or how it was described",
                              r"today'?s session|session today|today is an|today was an|easy (run|session|exercise|jog)|it'?s tempo|tempo run scheduled"
                              r"|session (would|involved|was)|type of session|session type|the level (and the|of) effort|intervals"),
            "FOCUS_TODAY": ("Focuses only on today", r"focus(ed)? on what i had to do today|focused on the current day|only think about today"),
        },
    },
    "q_tracking": {
        "single_name": None, "single": {},
        "multi": {
            "MONITORING": ("Watching time, pace, or distance", r"(?<!manage my )\btime\b(?! management)|timing|\bpace\b|tempo|minutes|\d+ ?min|countdown"),
            "MINIMAL": ("Little or nothing to keep track of", r"nothing much|nothing really|nothing except|^none\b|not much|wala naman|relatively chill"),
            "FOLLOW_PLAN": ("Following the plan's instructions or prescription", r"follow the plan|as prescribed|what was prescribed|prescribed time|instructions|type of session so"),
            "SESSION_STRUCTURE": ("Keeping track of the workout's structure (intervals, recoveries, switches)",
                                  r"interval|recovery|rounds|switching|when to rest|each section"),
            "BODY": ("Monitoring the body, effort, breathing, or energy", r"breathing|energy levels|my body|how i was feeling|level of effort i was|not pushing myself|handle the session"),
            "OUTSIDE_DEMANDS": ("Fitting the run around school, work, schedule, or health",
                                r"tight schedule|time management|right after|earlier in the morning|school|deadline|\bwork\b|on my period|unwell|sick"),
            "MOTIVATION": ("Showing up, keeping it up, or finishing the plan", r"show up|finish the (training )?plan|keep this up|good for my health"),
            "LOOK_AHEAD": ("Looking at the rest of the plan to plan ahead", r"seeing all the sessions|plan ahead|rest of the week|whole plan|looked ahead"),
            "UNCERTAIN_UPCOMING": ("Wondering what the unknown upcoming sessions will be", r"curious about what i should expect|unsure of what to expect|(didn't|did not|don't) know what"),
            "PLAN_ACCESS": ("Finding, re-reading, or checking the session instructions", r"looked at the plan|check(ed)? the plan|find the (message|plan|session)|re-?read"),
            "LOGISTICS": ("Practical preparation (clothes, getting ready, route, water)", r"getting ready|cleaning up|clothes|\bwater\b|route"),
        },
    },
    "q_plan_mind": {
        "single_name": "valence",
        "single": {
            "MIXED": ("Both easier and harder", r"(easier|manageable|motivated).*(harder|challenging)|(harder|challenging).*(easier|manageable|motivated)"),
            "HARDER": ("Plan thoughts made the run harder", r"harder|tougher|challenging|rushed|tight schedule|more tired than"),
            "EASIER": ("Plan thoughts made the run easier",
                       r"easier|easy since|somewhat easy|less pressure|more relaxed|less discouraged|feasible|very helpful|helps push|rewarding|happy to|excited"),
            "NONE": ("No effect", r"^(no|none|nothing|not really|neither|nothing in particular|nope)\b"),
        },
        "multi": {
            "SESSION_FRAMING": ("How the session was framed (easy, time-based, walk breaks, recoveries, session type)",
                                r"easy run|as easy|easy metric|time-based|\d+[- ]minutes?|walk breaks|recovery periods|set interval|session type"
                                r"|instructions|too literally|expecting an? (easy|exhausting)"),
            "OUTSIDE_DEMANDS": ("School, work, deadlines, schedule, fatigue, or health", r"deadline|school|\bwork\b|hurry|tight schedule|review|limited time|fatigued|on my period|due today"),
            "STRUCTURE_MOTIVATION": ("Having a scheduled plan as a reason to run", r"scheduled run|sundan na plan|plan to follow|get off bed|keep me motivated"),
            "SOCIAL": ("Running with others", r"friend"),
            "ANTICIPATING": ("Looking forward to or thinking about upcoming sessions", r"looking forward to the|upcoming|next (session|run)"),
            "SURPRISE": ("A session felt different than expected", r"than (what )?i expected|wasn't expecting|caught me off guard"),
        },
    },
}
CODED_QUESTIONS = list(CODEBOOK)


def rule_code(text, question):
    """Apply the codebook's patterns to one answer. Returns {single label, theme flags}."""
    book = CODEBOOK[question]
    t = (text or "").strip().lower()
    out = {}
    if book["single_name"]:
        label = next((code for code, (_, pat) in book["single"].items() if re.search(pat, t)), None)
        if label is None:
            label = list(book["single"])[-1] if not is_substantive(text) else "UNCODED"
        out[book["single_name"]] = label
    for code, (_, pat) in book["multi"].items():
        out[code] = bool(re.search(pat, t))
    return out


def code_responses_rules(text_df):
    rows = []
    for _, r in text_df[text_df["question"].isin(CODED_QUESTIONS)].iterrows():
        rows.append({"response_id": r["response_id"], "question": r["question"], **rule_code(r["response"], r["question"])})
    return pd.DataFrame(rows)


def build_llm_prompt(question, items):
    """items: list of (response_id, text). No participant code or condition is ever included."""
    book = CODEBOOK[question]
    lines = [
        "You are a qualitative coder for a study about runners following a training plan.",
        f'Each answer below responds to the diary question: "{TEXT_QS[question]}"',
        "Apply ONLY the codes defined here. Code what the text says, not what you guess the runner meant.", ""]
    if book["single_name"]:
        lines.append(f'"{book["single_name"]}": choose EXACTLY ONE of:')
        lines += [f"  - {c}: {d}" for c, (d, _) in book["single"].items()]
        lines.append("  - UNCODED: none of the above fit")
    lines.append('"themes": choose ZERO OR MORE of:')
    lines += [f"  - {c}: {d}" for c, (d, _) in book["multi"].items()]
    keys = (f'"{book["single_name"]}": "<CODE>", ' if book["single_name"] else "") + '"themes": ["<CODE>", ...]'
    lines += ["", "Reply with ONLY a JSON array, one object per answer, in this exact form:",
              f'[{{"id": "<id>", {keys}, "evidence": "<short quote that justifies the codes>"}}]', "", "ANSWERS:"]
    lines += [f"{rid}: {json.dumps(txt)}" for rid, txt in items]
    return "\n".join(lines)


def parse_llm_response(reply, question, expected_ids):
    """Validate an LLM reply against the codebook. Returns (coded DataFrame, list of problems)."""
    book = CODEBOOK[question]
    problems = []
    m = re.search(r"\[.*\]", reply, re.S)
    if not m:
        return pd.DataFrame(), ["no JSON array found in reply"]
    try:
        items = json.loads(m.group(0))
    except json.JSONDecodeError as e:
        return pd.DataFrame(), [f"invalid JSON: {e}"]
    rows, seen = [], set()
    valid_single = set(book["single"]) | {"UNCODED"}
    for it in items:
        rid = str(it.get("id", "")).strip()
        if rid not in expected_ids or rid in seen:
            problems.append(f"unexpected or repeated id {rid!r}")
            continue
        seen.add(rid)
        row = {"response_id": rid, "question": question}
        if book["single_name"]:
            lab = str(it.get(book["single_name"], "")).upper().strip()
            if lab not in valid_single:
                problems.append(f"{rid}: invalid {book['single_name']} {lab!r} -> UNCODED")
                lab = "UNCODED"
            row[book["single_name"]] = lab
        themes = {str(t).upper().strip() for t in it.get("themes", [])}
        for bad in themes - set(book["multi"]):
            problems.append(f"{rid}: unknown theme {bad!r} dropped")
        row.update({c: c in themes for c in book["multi"]})
        row["evidence"] = it.get("evidence", "")
        rows.append(row)
    problems += [f"missing id {i}" for i in sorted(set(expected_ids) - seen)]
    return pd.DataFrame(rows), problems


def call_llm(prompt):
    if LLM_PROVIDER == "anthropic":
        import anthropic
        msg = anthropic.Anthropic().messages.create(model=LLM_MODEL, max_tokens=8000, temperature=0,
                                                    messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text
    if LLM_PROVIDER == "openai":
        from openai import OpenAI
        res = OpenAI().chat.completions.create(model=LLM_MODEL, temperature=0,
                                               messages=[{"role": "user", "content": prompt}])
        return res.choices[0].message.content
    return None


def run_llm_coding(text_df):
    coded, status = [], []
    for q in CODED_QUESTIONS:
        sub = text_df[text_df["question"] == q]
        for b, start in enumerate(range(0, len(sub), LLM_BATCH_SIZE), 1):
            batch = sub.iloc[start:start + LLM_BATCH_SIZE]
            ids = list(batch["response_id"])
            prompt_path = LLM_DIR / f"{q}_batch{b:02d}_prompt.txt"
            reply_path = LLM_DIR / f"{q}_batch{b:02d}_reply.txt"
            prompt_path.write_text(build_llm_prompt(q, list(zip(ids, batch["response"]))), encoding="utf-8")
            if not reply_path.exists() and LLM_PROVIDER != "manual":
                reply_path.write_text(call_llm(prompt_path.read_text(encoding="utf-8")), encoding="utf-8")
            if reply_path.exists():
                df, problems = parse_llm_response(reply_path.read_text(encoding="utf-8"), q, set(ids))
                coded.append(df)
                status.append(dict(batch=prompt_path.name, answers=len(ids), coded=len(df), problems="; ".join(problems)))
            else:
                status.append(dict(batch=prompt_path.name, answers=len(ids), coded=0, problems="waiting for reply file"))
    return (pd.concat(coded, ignore_index=True) if coded else pd.DataFrame()), pd.DataFrame(status)

## 5. Pipeline tests
Three layers of testing, all run automatically every time the notebook runs. If any test fails, the notebook stops here so no analysis is built on broken data.

1. **Unit tests**: each cleaning, preparation, statistics, and coding function on small hand-made inputs with known correct answers (including edge cases such as a 2-hour boundary, midnight crossings, and malformed LLM replies).
2. **Known-answer tests**: the synthetic data generator documents exactly which problems it planted (`SYNTHETIC_README.md`). The pipeline must find **exactly** those, no more and no fewer.
3. **Invariant tests**: properties the final tables must always have (one row per scheduled session, values in range, no identifying fields, etc.).

In [ ]:
BASE_ROW = {
    "timestamp": "11/9/2026 8:00:00", "email": "r01.dummy@example.com", "participant_code": "R01",
    "session_date": "11/9/2026", "finish_time": "7:30:00 AM", "week": "Week 1", "session": "Session 1",
    "session_type": "Easy run", "status": "As prescribed", "modify_reason": "", "skip_reason": "", "skip_note": "",
    "rpe": "11 - Light", "q_pacing": "No", **{c: "3" for c in TLX_COLS},
    "q_tracking": "Just the time on my watch.", "q_plan_mind": "Nothing", "q_other": "", "q_general": "",
}
SKIPPED = {"status": "Skipped", "skip_reason": "Weather", "rpe": "", "q_pacing": "", **{c: "" for c in TLX_COLS},
           "q_tracking": "", "q_plan_mind": ""}


def on_day(day, session, session_type=None, **extra):
    """A row for Week 1 `session`, run on November `day` and logged 30 minutes after finishing."""
    return {"session_date": f"11/{day}/2026", "timestamp": f"11/{day}/2026 8:00:00", "session": f"Session {session}",
            "session_type": session_type or PLAN[session - 1], **extra}


def mini_raw(*overrides):
    rows = [{**BASE_ROW, **o} for o in overrides]
    df = pd.DataFrame(rows, columns=list(BASE_ROW))
    df.insert(0, "row_id", np.arange(len(df)))
    return df


def run_clean(*overrides, **settings):
    lg = CleaningLog()
    old = {k: globals()[k] for k in settings}
    globals().update(settings)
    try:
        out = clean(mini_raw(*overrides), key, lg)
    finally:
        globals().update(old)
    return out, lg.frame()


def stops(*overrides):
    try:
        run_clean(*overrides)
    except FormCheckError:
        return True
    return False


def r01(df, session_index=1):
    return df[(df["participant_code"] == "R01") & (df["session_index"] == session_index)].iloc[0]

# ---------- Layer 1: unit tests ----------
def test_schema_maps_every_field():
    """Every question in the export is mapped to exactly one field (only the documented optional ones may be absent)."""
    mapped = set(match_columns(list(pd.read_csv(RAW_FILE, nrows=0).columns)).values())
    assert set(FIELD_PATTERNS) - mapped <= OPTIONAL_FIELDS, set(FIELD_PATTERNS) - mapped

def test_schema_finds_every_form_question():
    """Each Section 7 question on the live form maps to its own field (none is dropped as 'unexpected')."""
    qs = ["Did anything on your mind about the plan make today’s run feel easier or harder? Please describe.",
          "Is there anything else about today's session or the plan that stood out to you?",
          "Is there anything about this session in general you'd like to add?",
          "Is there anything else you'd like to add?"]
    others = [c for c in pd.read_csv(RAW_FILE, nrows=0).columns if _norm_header(c) not in map(_norm_header, qs)]
    mapping = match_columns(qs + others)
    assert [mapping.get(q) for q in qs] == ["q_plan_mind", "q_other", "q_general", "skip_note"]

def test_schema_rejects_missing_question():
    """A required question that is missing stops the pipeline with a clear error."""
    cols = [c for c in pd.read_csv(RAW_FILE, nrows=0).columns if not c.startswith("How hard did your body")]
    try:
        match_columns(cols)
    except ValueError as e:
        assert "rpe" in str(e)
    else:
        raise AssertionError("missing RPE column was not detected")

def test_schema_tolerates_curly_quotes_and_order():
    """Curly apostrophes (Google sometimes exports these) and reordered columns still map correctly."""
    cols = [c.replace("'", "’") for c in pd.read_csv(RAW_FILE, nrows=0).columns][::-1]
    assert set(FIELD_PATTERNS) - set(match_columns(cols).values()) <= OPTIONAL_FIELDS

def test_key_must_match_form_codes():
    """The hand-typed participant key must list exactly R01-R10 once each, with valid condition and group."""
    for broken in (key.iloc[1:], key.assign(participant_code=key["participant_code"].replace("R01", "r1")),
                   pd.concat([key, key.iloc[[0]]]), key.assign(condition=key["condition"].replace("Full Plan", "Full"))):
        try:
            check_key(broken)
        except ValueError:
            continue
        raise AssertionError("a broken key was accepted")
    check_key(key)

def test_rpe_labels_parsed():
    """RPE options with and without labels are read as numbers."""
    vals = ["13 - Somewhat hard", "12", "6 - No exertion at all", "20 - Maximal exertion"]
    df, _ = run_clean(*[{"session": f"Session {i + 1}", "session_type": PLAN[i], "rpe": v} for i, v in enumerate(vals)])
    got = df[df["participant_code"] == "R01"].sort_values("session_index")["rpe"].iloc[:4].tolist()
    assert got == [13, 12, 6, 20], got

def test_answer_outside_form_options_stops():
    """Answers the form cannot produce stop the pipeline instead of being silently fixed."""
    impossible = [{"tlx_mental": "11"}, {"tlx_mental": "7.5"}, {"rpe": "25"}, {"participant_code": "R99"},
                  {"participant_code": "r1"}, {"status": ""}, {"week": f"Week {N_WEEKS + 1}"},
                  {"session_type": "Hill run"}, {"tlx_effort": ""}, {"timestamp": "2026-11-09 08:00"},
                  {"status": "Modified"}, {"modify_reason": "Cut it short"}]
    assert [o for o in impossible if not stops(o)] == []
    assert not stops({})

def test_branching_respected():
    """A skipped session cannot carry ratings, and a completed one cannot carry a skip reason."""
    assert stops({"status": "Skipped", "skip_reason": "Weather"})       # still has the ratings from BASE_ROW
    assert stops({"skip_reason": "Weather"})
    df, _ = run_clean(SKIPPED)
    assert r01(df)["status"] == "skipped" and pd.isna(r01(df)["rpe"])

def test_other_skip_reason_kept():
    """A reason typed under 'Other:' becomes Other, with the text kept."""
    df, _ = run_clean({**SKIPPED, "skip_reason": "Family event"})
    assert r01(df)["skip_reason"] == "Other" and r01(df)["skip_reason_other"] == "Family event"

def test_late_threshold_boundary():
    """Exactly 2 h is on time; 2 h 1 min is late."""
    df, _ = run_clean({"timestamp": "11/9/2026 9:30:00"},
                      {"timestamp": "11/10/2026 9:31:00", "session_date": "11/10/2026", "session": "Session 2",
                       "session_type": "Interval run"})
    assert r01(df, 1)["submit_lag_h"] == 2.0 and not r01(df, 1)["flag_late"]
    assert r01(df, 2)["flag_late"]

def test_lag_across_midnight():
    """A run finished at 11:30 PM and logged at 12:30 AM the next day has a 1-hour lag."""
    df, _ = run_clean({"finish_time": "11:30:00 PM", "timestamp": "11/10/2026 0:30:00"})
    assert abs(r01(df)["submit_lag_h"] - 1.0) < 1e-9

def test_negative_lag_flagged():
    """A finish time later than the submission (e.g. PM picked instead of AM) is flagged and its lag is not used."""
    df, _ = run_clean({"finish_time": "7:30:00 PM", "timestamp": "11/9/2026 8:00:00"})
    assert r01(df)["flag_negative_lag"] and pd.isna(r01(df)["submit_lag_h"])

def test_duplicates_policy():
    """Duplicate entries for one session on the same date: the policy decides which is kept, and the difference is logged."""
    rows = ({"rpe": "11 - Light", "timestamp": "11/9/2026 8:00:00"}, {"rpe": "14", "timestamp": "11/9/2026 8:06:00"})
    last, lg = run_clean(*rows, DUPLICATE_POLICY="last")
    first, _ = run_clean(*rows, DUPLICATE_POLICY="first")
    assert r01(last)["rpe"] == 14 and r01(first)["rpe"] == 11
    assert lg["detail"].str.contains("rpe").any() and not r01(last)["flag_label_mismatch"]
    assert (last["participant_code"].eq("R01") & last["session_index"].eq(1)).sum() == 1

def test_duplicate_on_different_dates_flagged():
    """Two entries for one session on different dates are not a resubmission: the kept one is flagged."""
    df, lg = run_clean(on_day(9, 2), on_day(11, 2))
    assert r01(df, 2)["flag_label_mismatch"] and lg["issue"].str.contains("different dates").any()

def test_session_number_corrected_when_type_and_date_agree():
    """'Session 2' + 'Easy run' on Wednesday, when Session 2 (Interval) was already logged Tuesday and Session 3
    (Easy) is free, is moved to Session 3."""
    df, lg = run_clean(on_day(9, 1), on_day(10, 2), on_day(11, 2, "Easy run"), on_day(12, 4))
    assert r01(df, 3)["session_date"] == pd.Timestamp("2026-11-11") and r01(df, 2)["session_date"] == pd.Timestamp("2026-11-10")
    assert not df["flag_label_mismatch"].any() and lg["action"].str.contains("moved to Session 3").any()

def test_type_mismatch_flagged_not_moved():
    """If the session number is not contradicted by another entry, a wrong type is flagged and the plan's type is used."""
    df, _ = run_clean({"session_type": "Long run"})
    assert r01(df)["session_type"] == "Easy run" and r01(df)["flag_label_mismatch"]

def test_date_in_wrong_week_flagged():
    """A Week 2 entry dated in Week 1 is flagged."""
    df, _ = run_clean({"week": "Week 2"})
    assert r01(df, 6)["flag_date_problem"]

def test_missing_versus_withdrawn():
    """Unlogged sessions stay 'missing' even at the end (the runner may have kept running); only a confirmed
    withdrawal becomes 'withdrawn', and a runner with no entries at all is 'did_not_start'."""
    rows = [{"week": f"Week {w}", "session": f"Session {s}", "session_type": PLAN[s - 1],
             "session_date": (STUDY_START + pd.Timedelta(days=7 * (w - 1) + s - 1)).strftime("%m/%d/%Y"),
             "timestamp": (STUDY_START + pd.Timedelta(days=7 * (w - 1) + s - 1, hours=8)).strftime("%m/%d/%Y %H:%M:%S")}
            for w in range(1, N_WEEKS + 1) for s in range(1, 6)]
    r02 = [{**r, "participant_code": "R02", "email": "r02@example.com"} for r in rows[:-1]]
    df, _ = run_clean(*rows[:5], *r02)
    a = df[df["participant_code"] == "R01"]["status"].value_counts()
    b = df[df["participant_code"] == "R02"]["status"].value_counts()
    assert a.get("missing", 0) == len(rows) - 5 and a.get("withdrawn", 0) == 0
    assert b.get("missing", 0) == 1 and b.get("withdrawn", 0) == 0
    others = df[~df["participant_code"].isin(["R01", "R02"])]
    assert set(others["status"]) == {"did_not_start"}
    df2, _ = run_clean(*rows[:5], *r02, CONFIRMED_WITHDRAWALS={"R01": 5})
    a2 = df2[df2["participant_code"] == "R01"]["status"].value_counts()
    assert a2.get("withdrawn", 0) == len(rows) - 5 and a2.get("missing", 0) == 0

def test_email_typo_recoded_and_email_removed():
    """A known participant who picks someone else's code is recoded via their email, then emails are deleted."""
    rows = [{"session": f"Session {s}", "session_type": PLAN[s - 1]} for s in range(1, 6)]
    rows[3]["participant_code"] = "R05"
    df, _ = run_clean(*rows)
    submitted = df["status"].isin(["prescribed", "modified", "skipped"])
    assert (submitted & df["participant_code"].eq("R01")).sum() == 5
    assert not (submitted & df["participant_code"].eq("R05")).any()
    assert "email" not in df.columns

def test_wrong_code_flagged_when_too_few_entries():
    """With too few entries to recode, an entry under R02 sent from R01's email is flagged for checking."""
    df, _ = run_clean({}, on_day(10, 2, participant_code="R02"),
                      {"participant_code": "R02", "email": "r02@example.com"})
    row = df[df["participant_code"].eq("R02") & df["session_index"].eq(2)].iloc[0]
    assert row["flag_code_check"]

def test_manual_corrections_applied():
    """Corrections confirmed with the runner are applied and logged; test entries can be dropped."""
    fix = {("R05", "11/9/2026 8:00:00"): {"participant_code": "R01"}}
    df, lg = run_clean({"participant_code": "R05", "email": "r05@example.com"}, MANUAL_CORRECTIONS=fix)
    assert r01(df)["status"] == "prescribed" and lg["issue"].str.contains("confirmed").any()
    df, _ = run_clean({}, MANUAL_CORRECTIONS={("R01", "11/9/2026 8:00:00"): {"drop": "researcher test"}})
    assert not df.loc[df["participant_code"].eq("R01"), "submitted" if "submitted" in df else "status"].isin(
        ["prescribed", "modified", "skipped"]).any()

def test_placeholders_kept_as_no():
    """'N/A' or '-' typed into a required text question is kept and counted as a non-substantive answer."""
    df, lg = run_clean({"q_pacing": "N/A", "q_plan_mind": " - "})
    assert r01(df)["q_pacing"] == "N/A" and r01(df)["q_plan_mind"] == "-"
    assert lg["issue"].str.contains("placeholder").any() and not is_substantive("N/A") and not is_substantive("-")

def test_raw_tlx_scaling():
    """Raw TLX is the mean of six subscales on 0-100, and is blank if any subscale is missing."""
    df, _ = run_clean({}, on_day(10, 2))
    df.loc[df["participant_code"].eq("R01") & df["session_index"].eq(2), "tlx_effort"] = pd.NA
    s, _ = prepare(df, key)
    a, b = s[(s.participant_code == "R01") & (s.session_index == 1)].iloc[0], s[(s.participant_code == "R01") & (s.session_index == 2)].iloc[0]
    assert a["tlx_raw"] == 30 and a["mental"] == 30 and pd.isna(b["tlx_raw"])

def test_next_session_info():
    """Plan position: what comes next, and the final session."""
    assert next_session_info(1, 1) == ("Interval run", "harder session next")
    assert next_session_info(1, 2) == ("Easy run", "easy run next")
    assert next_session_info(1, 5) == ("Easy run", "easy run next")
    assert next_session_info(N_WEEKS, 5) == (None, "final session")

def test_is_substantive():
    """Terse answers ('No', 'Nothing.') are not substantive; real descriptions are."""
    assert not any(map(is_substantive, ["No", "Nothing.", "not really", "", None, "Neither", "N/A", "-", "."]))
    assert not any(map(is_substantive, ["Wala naman.", "Hindi po", "Wala", "Okay lang naman", "Hindi naman masyado",
                                        "di naman", "Ayos lang po."]))
    assert is_substantive("Wala naman masyadong iniisip, pero mas gumaan kasi isang session lang.")
    assert is_substantive("Easier, I only had to think about today.")

def test_detect_language():
    """English answers stay English; Filipino and Taglish answers (including 'nag-' forms) are tagged."""
    english = ["Easier, I only had to think about today.", "Just the time on my watch.", "No.",
               "I held back a bit because the tempo run was next."]
    filipino = ["Wala naman.", "Oo, nag-hold back ako kasi alam kong may tempo run bukas.",
                "Yung pace ko tsaka yung reps.", "Medyo bumigat kasi iniisip ko yung long run sa weekend.",
                "Mas gumaan, I only focused on today's run."]
    assert [detect_language(t) for t in english] == ["English"] * len(english)
    assert [detect_language(t) for t in filipino] == ["Filipino or Taglish"] * len(filipino)

def test_additive_residual_removes_both_effects():
    """If y is exactly runner effect + session-type effect, the residual is zero."""
    d = pd.DataFrame({"participant_code": list("AABBCC"), "session_type": ["x", "y"] * 3})
    d["y"] = d["participant_code"].map({"A": 1, "B": 5, "C": 9}) + d["session_type"].map({"x": 0, "y": 3})
    assert np.allclose(additive_residual(d, "y"), 0)

def test_rmcorr():
    """rmcorr equals Pearson r on within-runner centered data, uses N - k - 1 df, and ignores between-runner trends."""
    g = np.random.default_rng(1)
    d = pd.DataFrame({"participant_code": np.repeat(list("ABCD"), 6), "x": g.normal(size=24)})
    d["y"] = 0.8 * d["x"] - 5 * d["participant_code"].map(dict(zip("ABCD", range(4)))) + g.normal(0, 0.3, 24)
    d["x"] += 3 * d["participant_code"].map(dict(zip("ABCD", range(4))))   # between-runner trend is negative
    res = rmcorr(d, "x", "y")
    xc = d["x"] - d.groupby("participant_code")["x"].transform("mean")
    yc = d["y"] - d.groupby("participant_code")["y"].transform("mean")
    assert abs(res["r"] - np.corrcoef(xc, yc)[0, 1]) < 1e-12 and res["dof"] == 24 - 4 - 1
    assert res["r"] > 0.8 and stats.pearsonr(d["x"], d["y"]).statistic < 0

def test_hedges_g_known_value():
    """g for [1,2,3] vs [4,5,6] is -3 x (1 - 3/15) = -2.4."""
    assert abs(hedges_g([1, 2, 3], [4, 5, 6]) + 2.4) < 1e-12

def test_permutation_test():
    """Perfect separation gives the smallest possible p (2/400); identical groups give p = 1."""
    pm = pd.DataFrame({"participant_code": [f"P{i}" for i in range(12)],
                       "condition": ["Full Plan"] * 3 + ["Current Only"] * 3 + ["Full Plan"] * 3 + ["Current Only"] * 3,
                       "experience_group": ["Novice"] * 6 + ["Experienced"] * 6})
    pm["y"] = np.where(pm["condition"] == "Full Plan", 10.0, 0.0) + np.arange(12) * 0.01
    res = stratified_permutation_test(pm, "y")
    assert res["n_assignments"] == 400 and abs(res["p"] - 2 / 400) < 1e-12
    assert stratified_permutation_test(pm.assign(y=1.0), "y")["p"] == 1.0

def test_mixed_model_recovers_known_effect():
    """On simulated data with a true Full Plan effect of +2.0, the mixed model's 95% CI contains 2.0."""
    g = np.random.default_rng(7)
    sim = sessions[["participant_code", "condition", "experience_group", "week", "session_type", "completed"]].copy()
    sim["completed"] = True
    runner = dict(zip(sim["participant_code"].unique(), g.normal(0, 0.7, sim["participant_code"].nunique())))
    sim["rpe"] = (12 + 2.0 * sim["condition"].eq("Full Plan") + sim["participant_code"].map(runner)
                  + sim["session_type"].map({"Easy run": 0, "Interval run": 3, "Tempo run": 2, "Long run": 2.5}).astype(float)
                  + g.normal(0, 1, len(sim)))
    res = fit_mixed(sim, "rpe")
    assert res["ci_lo"] < 2.0 < res["ci_hi"], (res["ci_lo"], res["ci_hi"])

def test_holm():
    """Holm adjustment matches a hand-worked example."""
    assert np.allclose(holm([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06])

def test_cohen_kappa_matches_sklearn():
    """Our kappa matches scikit-learn's (when installed) and a hand-worked value."""
    a, b = list("AABBCCAB"), list("ABBBCCAA")
    try:
        from sklearn.metrics import cohen_kappa_score
        assert abs(cohen_kappa(a, b) - cohen_kappa_score(a, b)) < 1e-12
    except ImportError:
        pass
    assert cohen_kappa([True, False] * 5, [True, False] * 5) == 1.0

def test_rule_coder_examples():
    """The rule coder assigns the expected codes to typical answers."""
    p = rule_code("Yes, I saw the tempo run was next so I held back a bit today.", "q_pacing")
    assert p["pacing"] == "HELD_BACK" and p["KNEW_UPCOMING"] and not p["UNCERTAIN_UPCOMING"]
    p = rule_code("A bit. Not knowing the next session made me hold back on the last part.", "q_pacing")
    assert p["pacing"] == "HELD_BACK" and p["UNCERTAIN_UPCOMING"] and not p["KNEW_UPCOMING"]
    assert rule_code("No.", "q_pacing")["pacing"] == "NO_EFFECT"
    assert rule_code("It helped me prepare for the effort the session needed.", "q_pacing")["pacing"] == "PREPARED"
    m = rule_code("Harder. I wasn't expecting this long run today so I hadn't really prepared.", "q_plan_mind")
    assert m["valence"] == "HARDER" and m["SURPRISE"]
    m = rule_code("It was challenging, but the rest breaks made it more manageable.", "q_plan_mind")
    assert m["valence"] == "MIXED"
    t = rule_code("Had to find the message with today's session first.", "q_tracking")
    assert t["PLAN_ACCESS"] and not t["LOOK_AHEAD"]
    # Regression tests for the first codebook's misfires
    assert not rule_code("I got really tired near the end of the hard part.", "q_plan_mind")["OUTSIDE_DEMANDS"]
    assert not rule_code("I watched for the recovery periods between the fast parts.", "q_tracking")["OUTSIDE_DEMANDS"]
    t = rule_code("Seeing all the sessions helped me plan ahead and manage my time.", "q_tracking")
    assert t["LOOK_AHEAD"] and not t["OUTSIDE_DEMANDS"] and not t["MONITORING"]
    t = rule_code("Nothing much, though I'm curious about what I should expect in the next ones.", "q_tracking")
    assert t["UNCERTAIN_UPCOMING"] and not t["LOOK_AHEAD"] and t["MINIMAL"]
    assert rule_code("It felt harder because I was tired from school.", "q_plan_mind")["OUTSIDE_DEMANDS"]

def test_llm_reply_parser():
    """Malformed LLM replies are handled: code fences, invalid codes, unknown themes, missing and extra ids."""
    reply = """Sure! ```json
    [{"id": "T1", "pacing": "held_back", "themes": ["KNEW_UPCOMING", "MADE_UP"], "evidence": "held back"},
     {"id": "T2", "pacing": "SPRINTED", "themes": []},
     {"id": "T9", "pacing": "PUSHED", "themes": []}]
    ``` Hope this helps."""
    df, problems = parse_llm_response(reply, "q_pacing", {"T1", "T2", "T3"})
    assert list(df["response_id"]) == ["T1", "T2"]
    assert df.loc[0, "pacing"] == "HELD_BACK" and df.loc[0, "KNEW_UPCOMING"] and df.loc[1, "pacing"] == "UNCODED"
    joined = " ".join(problems)
    assert "MADE_UP" in joined and "SPRINTED" in joined and "T9" in joined and "missing id T3" in joined
    assert parse_llm_response("no json here", "q_pacing", {"T1"})[1] == ["no JSON array found in reply"]

def test_llm_prompt_is_blind():
    """LLM prompts never contain participant codes or condition names."""
    sample = text_responses[text_responses["question"] == "q_pacing"].head(30)
    prompt = build_llm_prompt("q_pacing", list(zip(sample["response_id"], sample["response"])))
    assert not re.search(r"\bR\d{2}\b", prompt) and "Full Plan" not in prompt and "Current Only" not in prompt

# ---------- Layer 2: known-answer tests against the synthetic ground truth ----------
TRUTH = {
    "withdrawn": {"R03": (2, 3)},
    "missing": {("R01", 1, 5), ("R01", 3, 5), ("R02", 3, 3), ("R07", 3, 1), ("R09", 1, 3), ("R09", 3, 1)},
    "late": {("R12", 1, 1), ("R09", 1, 2), ("R11", 1, 3), ("R05", 1, 3), ("R01", 1, 4), ("R08", 2, 3),
             ("R10", 3, 1), ("R11", 3, 4)},
    "duplicate": {("R04", 2, 3)},
}


def as_set(df):
    return set(zip(df["participant_code"], df["week"].astype(int), df["session"].astype(int)))


def test_truth_withdrawal():
    """Exactly R03 is detected as withdrawn, from Week 2 Session 4 onward."""
    w = sessions[sessions["status"] == "withdrawn"]
    assert set(w["participant_code"]) == {"R03"} and w["session_index"].min() == SESSIONS_PER_WEEK + 4

def test_truth_missing():
    """Exactly the six documented sessions have no form."""
    assert as_set(sessions[sessions["status"] == "missing"]) == TRUTH["missing"]

def test_truth_late():
    """Exactly the eight documented late entries are flagged."""
    assert as_set(sessions[sessions["flag_late"]]) == TRUTH["late"]

def test_truth_duplicate():
    """Exactly the one documented duplicate is found and resolved."""
    assert as_set(sessions[sessions["flag_duplicate_resolved"]]) == TRUTH["duplicate"]

def test_truth_effect_direction():
    """The built-in effects (Full Plan higher on mental demand and RPE) are recovered in direction."""
    pm = participant_means(sessions, ["mental", "rpe"])
    d = pm.groupby("condition", observed=True)[["mental", "rpe"]].mean()
    assert (d.loc["Full Plan"] > d.loc["Current Only"]).all()

# ---------- Layer 3: invariants of the final tables ----------
def test_invariant_one_row_per_scheduled_session():
    """sessions has exactly one row per participant x scheduled session."""
    assert len(sessions) == key["participant_code"].nunique() * N_WEEKS * SESSIONS_PER_WEEK
    assert not sessions.duplicated(["participant_code", "week", "session"]).any()

def test_invariant_ranges():
    """All ratings are within their scales."""
    assert sessions["rpe"].dropna().between(6, 20).all()
    for c in TLX_COLS:
        assert sessions[f"{c}_100"].dropna().between(0, 100).all()
    assert sessions["submit_lag_h"].dropna().ge(0).all()

def test_invariant_status_consistency():
    """Only completed sessions have ratings, and every completed session has all of them (the form requires them)."""
    assert sessions.loc[~sessions["completed"], ["rpe", "mental", "tlx_raw"]].isna().all().all()
    assert sessions.loc[sessions["completed"], ["rpe", "tlx_raw"]].notna().all().all()

def test_invariant_no_identifiers():
    """No names or emails survive into any output table."""
    for df in (sessions, participants, text_responses, tlx_long):
        assert not any("email" in c.lower() or "name" in c.lower() for c in df.columns)
        assert not df.astype(str).apply(lambda s: s.str.contains("@", regex=False)).any().any()

def test_invariant_balanced_design():
    """Each condition has the same number of runners in each experience group (stratified assignment)."""
    if not IS_SYNTHETIC:
        return   # real recruitment ended unbalanced (6 FP / 4 CO); this is reported in the paper instead
    t = participants.groupby(["condition", "experience_group"], observed=True).size()
    assert t.nunique() == 1

def test_invariant_exports_round_trip():
    """Exported CSVs re-load with the same shape."""
    assert pd.read_csv(OUT_DIR / "sessions.csv").shape == sessions.shape
    assert pd.read_csv(OUT_DIR / "text_responses.csv").shape == text_responses.shape

def run_tests(namespace):
    results = []
    for name, fn in list(namespace.items()):
        if not (name.startswith("test_") and callable(fn)):
            continue
        if name.startswith("test_truth") and not IS_SYNTHETIC:
            results.append((name, "SKIP", "real data: no ground truth", fn.__doc__))
            continue
        try:
            fn()
            results.append((name, "PASS", "", fn.__doc__))
        except AssertionError as e:
            results.append((name, "FAIL", str(e)[:200], fn.__doc__))
        except Exception as e:
            results.append((name, "ERROR", f"{type(e).__name__}: {e}"[:200], fn.__doc__))
    out = pd.DataFrame(results, columns=["test", "result", "message", "what it checks"])
    out.insert(0, "layer", np.select([out["test"].str.startswith("test_truth"), out["test"].str.startswith("test_invariant")],
                                     ["2 known-answer", "3 invariant"], "1 unit"))
    return out


test_results = run_tests(globals())
print(test_results["result"].value_counts().to_string())
assert test_results["result"].isin(["PASS", "SKIP"]).all(), "Some tests failed: see table"
test_results[["layer", "test", "result", "what it checks", "message"]]

## 6. Adherence and data quality
Before comparing conditions, check that both groups actually followed the plan and logged it. If one condition skipped or modified more sessions, differences in ratings could come from that instead of disclosure pacing.

In [ ]:
def adherence_table(sessions):
    s = sessions[~sessions["status"].isin(NOT_EXPECTED)]
    t = pd.crosstab(s["condition"], s["status"], normalize="index").reindex(
        columns=["prescribed", "modified", "skipped", "missing"], fill_value=0) * 100
    t.columns = [f"% {c}" for c in t.columns]
    t.insert(0, "scheduled sessions", s.groupby("condition", observed=True).size())
    done = s[s["completed"]]
    t["% of entries late (>2 h)"] = done.groupby("condition", observed=True)["flag_late"].mean() * 100
    t["median lag (h)"] = done.groupby("condition", observed=True)["submit_lag_h"].median()
    t["withdrawals (confirmed)"] = sessions[sessions["status"] == "withdrawn"].groupby("condition", observed=True)["participant_code"].nunique()
    p = participants.set_index("participant_code")
    t["stopped logging early"] = p[p["stopped_early"] & ~p["did_not_start"]].groupby("condition", observed=True).size()
    t["did not start"] = p[p["did_not_start"]].groupby("condition", observed=True).size()
    return t.fillna(0).round(1)


adherence_table(sessions)

In [ ]:
def plot_compliance_calendar(sessions):
    order = (sessions[["participant_code", "condition"]].drop_duplicates()
             .sort_values(["condition", "participant_code"]))
    codes = order["participant_code"].tolist()
    status_order = STATUS_ORDER
    glyph = {"prescribed": "✓", "modified": "M", "skipped": "S", "missing": "?", "withdrawn": "", "did_not_start": "–"}
    glyph_color = {"prescribed": "white", "modified": INK, "skipped": "white", "missing": INK2, "withdrawn": INK2,
                   "did_not_start": MUTED}
    mat = sessions.pivot(index="participant_code", columns="session_index", values="status").loc[codes].astype(str)
    late = sessions.pivot(index="participant_code", columns="session_index", values="flag_late").loc[codes]
    num = mat.apply(lambda c: c.map({s: i for i, s in enumerate(status_order)})).to_numpy(float)
    n_rows, n_cols = num.shape
    fig, ax = plt.subplots(figsize=(11.5, 0.42 * n_rows + 2.2))
    ax.imshow(num, cmap=ListedColormap([STATUS_COLORS[s] for s in status_order]), vmin=-0.5, vmax=len(status_order) - 0.5, aspect="auto")
    ax.set_xticks(np.arange(-.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-.5, n_rows, 1), minor=True)
    ax.grid(which="minor", color=SURFACE, linewidth=2)
    ax.grid(which="major", visible=False)
    ax.tick_params(which="both", length=0)
    for w in range(1, N_WEEKS):
        ax.axvline(w * SESSIONS_PER_WEEK - 0.5, color=SURFACE, lw=7)
    n_fp = int((order["condition"] == "Full Plan").sum())
    ax.axhline(n_fp - 0.5, color=SURFACE, lw=7)
    for i in range(n_rows):
        for j in range(n_cols):
            st = mat.iat[i, j]
            ax.text(j, i, glyph[st], ha="center", va="center", fontsize=9, color=glyph_color[st], fontweight="bold")
            if late.iat[i, j]:
                ax.plot(j + 0.3, i - 0.27, marker="o", ms=4, color=INK, mec=SURFACE, mew=0.8)
    short = {"Easy run": "Easy", "Interval run": "Int.", "Tempo run": "Tempo", "Long run": "Long"}
    ax.set_xticks(range(n_cols), [short[PLAN[k % SESSIONS_PER_WEEK]] for k in range(n_cols)], fontsize=8)
    ax.set_yticks(range(n_rows), codes)
    for w in range(N_WEEKS):
        ax.text(w * SESSIONS_PER_WEEK + 2, -0.85, f"Week {w + 1}", ha="center", va="bottom", color=INK2, fontsize=9)
    for cond, (lo, hi) in {"Full Plan": (0, n_fp - 1), "Current Only": (n_fp, n_rows - 1)}.items():
        ax.text(-1.55, (lo + hi) / 2, cond, rotation=90, ha="center", va="center", color=COND_COLORS[cond],
                fontweight="bold")
    for sp in ax.spines.values():
        sp.set_visible(False)
    handles = [Patch(facecolor=STATUS_COLORS[s], label=f"{glyph[s] or '·'}  {s.replace('_', ' ')}") for s in status_order]
    handles.append(Line2D([], [], marker="o", ls="", color=INK, ms=5, label="entry > 2 h late"))
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=7, handlelength=1.2)
    ax.set_title("Who ran what: session status for every runner and scheduled session", pad=34)
    fig.tight_layout()
    save_fig(fig, "fig01_compliance_calendar")
    return fig


plot_compliance_calendar(sessions);

In [ ]:
def plot_submission_lag(sessions):
    d = sessions[sessions["completed"]].dropna(subset=["submit_lag_h"])
    fig, ax = plt.subplots(figsize=(9, 2.8))
    g = np.random.default_rng(SEED)
    for k, cond in enumerate(CONDITIONS):
        v = d.loc[d["condition"] == cond, "submit_lag_h"]
        ax.scatter(v, k + g.uniform(-0.18, 0.18, len(v)), s=22, color=COND_COLORS[cond], alpha=0.75,
                   edgecolor=SURFACE, linewidth=0.6)
        ax.text(d["submit_lag_h"].max() * 1.08, k, f"{(v > LATE_THRESHOLD_H).sum()} late of {len(v)}",
                va="center", color=INK2, fontsize=9)
    ax.axvline(LATE_THRESHOLD_H, color=INK2, lw=1.2)
    ax.text(LATE_THRESHOLD_H + 0.05, 1.45, "2-hour window", color=INK2, fontsize=9, va="center")
    ax.set_yticks([0, 1], CONDITIONS)
    ax.set_ylim(-0.6, 1.7)
    ax.set_xlim(0, d["submit_lag_h"].max() * 1.32)
    ax.set_xlabel("Hours between finishing the run and submitting the diary")
    ax.grid(axis="y", visible=False)
    ax.set_title("Diary timing: most entries arrive within the 2-hour window")
    fig.tight_layout()
    save_fig(fig, "fig02_submission_lag")
    return fig


plot_submission_lag(sessions);

In [ ]:
skips = sessions[sessions["status"] == "skipped"]
pd.crosstab(skips["skip_reason"], skips["condition"], margins=True, margins_name="Total")

## 7. RQ1: Cognitive load (NASA-TLX)
**Primary analysis (descriptive, per the proposal):** each runner's average across completed sessions, summarized per condition. Mental demand is the main indicator of plan-related load.

In [ ]:
TLX_LABELS = {"tlx_raw": "Raw TLX", "mental": "Mental demand", "tlx_physical_100": "Physical demand",
              "tlx_temporal_100": "Temporal demand", "tlx_performance_100": "Performance (0 = perfect)",
              "tlx_effort_100": "Effort", "tlx_frustration_100": "Frustration"}
pm_all = participant_means(sessions, ["rpe", *TLX_LABELS])
describe_by_condition(pm_all, list(TLX_LABELS), TLX_LABELS).round(1)

In [ ]:
def plot_tlx_profile(pm):
    rows = list(TLX_LABELS)[::-1]
    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    g = np.random.default_rng(SEED)
    for y, col in enumerate(rows):
        if col == "mental":
            ax.axhspan(y - 0.45, y + 0.45, color="#eef4fc", zorder=0)
        means = {}
        for k, cond in enumerate(CONDITIONS):
            v = pm.loc[pm["condition"] == cond, col].dropna()
            off = 0.13 if k == 0 else -0.13
            ax.scatter(v, y + off + g.uniform(-0.05, 0.05, len(v)), s=16, color=COND_COLORS[cond], alpha=0.45,
                       linewidth=0, zorder=2)
            means[cond] = (v.mean(), y + off)
        (x1, y1), (x2, y2) = means["Full Plan"], means["Current Only"]
        ax.plot([x1, x2], [y1, y2], color=AXIS, lw=2, zorder=1)
        for cond, (x, yy) in means.items():
            ax.scatter(x, yy, s=90, color=COND_COLORS[cond], edgecolor=SURFACE, linewidth=2, zorder=3)
        ax.text(102, y, f"{x1 - x2:+.1f}", va="center", ha="left", color=INK2, fontsize=9)
    ax.text(102, len(rows) - 0.35, "FP − CO", ha="left", color=MUTED, fontsize=8)
    ax.set_yticks(range(len(rows)), [TLX_LABELS[r] for r in rows])
    ax.get_yticklabels()[rows.index("mental")].set_fontweight("bold")
    ax.set_xlim(0, 100)
    ax.set_xlabel("Score, 0-100 (higher = more workload)")
    ax.grid(axis="y", visible=False)
    ax.legend(handles=[Line2D([], [], marker="o", ls="", ms=9, color=COND_COLORS[c], label=f"{c} (mean)") for c in CONDITIONS]
              + [Line2D([], [], marker="o", ls="", ms=5, color=MUTED, alpha=.6, label="one runner's average")],
              loc="lower right", fontsize=9)
    ax.set_title("RQ1 · Workload profile by condition (mental demand = main indicator)")
    fig.tight_layout()
    save_fig(fig, "fig03_tlx_profile")
    return fig


plot_tlx_profile(pm_all);

**Mental demand by session type.** If Full Plan runners carry more plan-related load, the gap should appear across session types, not only on complex sessions such as intervals.

In [ ]:
by_type = (sessions[sessions["completed"]]
           .groupby(["participant_code", "condition", "session_type"], observed=True)["mental"].mean().reset_index()
           .groupby(["session_type", "condition"], observed=True)["mental"].agg(["mean", "std", "count"]).round(1)
           .unstack("condition"))
by_type.columns = [f"{stat} · {cond}" for stat, cond in by_type.columns]
by_type

## 8. RQ2: Perceived exertion (Borg RPE)

This section contains the summary statistics for the Borg Rating of Perceived Excertion (Borg RPE) and the statistical analysis to determine:

- How does the pacing of exercise disclosure affect runners' rate of perceived exertion (RPE) during a structured run?

A Linear Mixed Model (LMM) is used to to measure how a workout plan's disclosure method `(condition)`, session type `(session_type)`, running experience `(experience_group)`, and time `(session_index)` affect the runner's perceived exertion `(rpe)`, while accounting for repeated measurements from the same participant using `(participant_code)`.

### Summary Statistics for Borg RPE, grouped by disclosure method


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

def plot_summary_statistics(df):
    # Create figure with 2 side-by-side subplots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 1. Box & Whisker Plot (Updated for Seaborn >= 0.13)
    sns.boxplot(
        data=df,
        x="condition",
        y="rpe",
        hue="condition",  # Assign hue to match x variable
        palette="Set2",
        legend=False,  # Prevents redundant legend boxplot key
        showmeans=True,  # Shows mean as a distinct marker alongside median
        meanprops={
            "marker": "o",
            "markerfacecolor": "white",
            "markeredgecolor": "black",
        },
        ax=axes[0],
    )
    axes[0].set_title("RPE Distribution by Condition", fontsize=12)
    axes[0].set_xlabel("Condition", fontsize=10)
    axes[0].set_ylabel("RPE", fontsize=10)

    # 2. Histogram with Kernel Density Estimate (shows Skewness)
    sns.histplot(
        data=df,
        x="rpe",
        hue="condition",
        kde=True,
        element="step",
        common_norm=False,
        palette="Set2",
        ax=axes[1],
    )
    axes[1].set_title("RPE Skewness & Frequency Histogram", fontsize=12)
    axes[1].set_xlabel("RPE", fontsize=10)
    axes[1].set_ylabel("Count", fontsize=10)

    plt.tight_layout()
    save_fig(fig, "fig11_rpe_runner_averages")
    plt.show()

### Summary Statistics for an Individual Runner's average RPE across their completed sessions

In [ ]:
plot_summary_statistics(pm_all)
describe_by_condition(pm_all, ["rpe"], {"rpe": "Borg RPE (6-20)"}).round(2)


### Summary Statistics for RPE by condition

In [ ]:
# 1. Merge key dataframe into clean_df on participant_code
key["participant_code"] = key["participant_code"].astype(str).str.strip()
clean_df["participant_code"] = clean_df["participant_code"].astype(str).str.strip()

rpe_df = clean_df.merge(
    key[["participant_code", "condition", "experience_group"]],
    on="participant_code",
    how="left",
)

# 2. Compute summary statistics using named aggregation
summary_stats = (
    rpe_df.groupby("condition")["rpe"]
    .agg(
        Count="count",
        Missing=lambda x: x.isna().sum(),
        Missing_pct=lambda x: (x.isna().sum() / len(x)) * 100,
        Mean="mean",
        Std="std",
        Min="min",
        Q25=lambda x: x.quantile(0.25),
        Median="median",
        Q75=lambda x: x.quantile(0.75),
        Max="max",
        IQR=lambda x: x.quantile(0.75) - x.quantile(0.25),
        Skewness="skew",
    )
    .reset_index()
    .rename(columns={"Missing_pct": "Missing_%"})
)

plot_summary_statistics(pm_all)   # one average per runner (the planned unit of analysis)
fig = plt.figure(figsize=(7, 4))
sns.histplot(data=rpe_df.dropna(subset=["rpe"]), x="rpe", hue="condition", discrete=True, multiple="dodge", palette="Set2")
plt.title("Session-level RPE by condition")
plt.tight_layout()
save_fig(fig, "fig12_rpe_sessions")
plt.show()
display(summary_stats)

### Summary Statistics for RPE by condition, faceted by session type

In [ ]:
# Stratify by session_type and condition simultaneously
session_sensitivity = (
    rpe_df.groupby(["session_type", "condition"])["rpe"]
    .agg(Count="count", Mean="mean", Std="std", Median="median")
    .reset_index()
)

# Faceted Boxplot Grid
g = sns.catplot(
    data=rpe_df,
    x="condition",
    y="rpe",
    col="session_type",
    hue="condition",  # Set hue equal to x to prevent warning
    kind="box",
    palette="Set2",
    legend=False,
    height=4,
    aspect=0.9,
    showmeans=True,
    meanprops={
        "marker": "o",
        "markerfacecolor": "white",
        "markeredgecolor": "black",
    },
)

g.set_axis_labels("Condition", "RPE")
g.set_titles(col_template="{col_name}")
g.fig.subplots_adjust(top=0.82)
g.fig.suptitle("RPE by Condition Faceted by Session Type", fontsize=13)
save_fig(g.fig, "fig13_rpe_by_session_type")

plt.show()

### Linear Mixed Models

Linear Mixed Models (LMMs) are used to measure how a workout plan's disclosure method `(condition)`, workout type `(session_type)`, and time `(session_index)` affect the runner's perceived exertion `(rpe)`, while accounting for repeated measurements from the same participant using `participant_code`.

`experience_group` is **not** included: among runners with diary data, only one is Experienced and none of the Current Only runners are, so its effect cannot be estimated (the models fail with a singular matrix). The imbalance is reported as a limitation instead.

| Model Object | Formula | What It Measures | Relevance |
|---|---|---|---|
| `fit_main` | `rpe ~ C(condition) + C(session_type)` | The association of condition and workout type with RPE, each accounting for the other. | "Does Full Plan change RPE overall, accounting for workout type?" |
| `fit_session` | `rpe ~ C(condition) * C(session_type)` | Whether the association between condition and RPE varies depending on workout type. | "Does Full Plan affect RPE differently depending on workout type?" |
| `fit_time` | `rpe ~ C(condition) * session_index + C(session_type)` | Whether the difference in RPE between conditions changes across successive sessions (condition-specific time trend). | "Does the relationship between training condition and RPE change over the 10 sessions?" |

With only 8 runners (5 Full Plan, 3 Current Only), these models are exploratory. A model that cannot be fitted is reported and skipped instead of stopping the notebook.


The following variables are used as the baseline for their respective columns:

- `condition`: "Current Only"
- `session_type`: "Easy run"

Meanwhile, `session_index` is numerical. This value ranges from 1 to N, where N is the number of sessions required in the study. Each runner has its own session index, ordered by date. Not all sessions are guaranteed to have a value for its `rpe`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.formula.api as smf

# 1. Prepare dataframe
analysis_df = rpe_df.dropna(subset=["rpe"]).copy()
analysis_df["condition"] = pd.Categorical(
    analysis_df["condition"], categories=["Current Only", "Full Plan"]
)

# 2. Fit the LMMs (random intercept per participant).
# experience_group is left out: too few Experienced runners with data to estimate it (see the note above).
formulas = {
    "Primary Model": "rpe ~ C(condition) + C(session_type)",
    "Session Model": "rpe ~ C(condition) * C(session_type)",
    "Time Model":    "rpe ~ C(condition) * session_index + C(session_type)",
}
models = {}
for name, formula in formulas.items():
    try:
        models[name] = smf.mixedlm(formula, data=analysis_df, groups=analysis_df["participant_code"]).fit(reml=True)
        print(f"\n=================== {name.upper()} RESULTS ===================")
        print(models[name].summary())
    except Exception as e:
        print(f"\n{name}: could not be fitted ({type(e).__name__}: {e})")

fit_main = models.get("Primary Model")
fit_session = models.get("Session Model")
fit_time = models.get("Time Model")

# Fixed effects table with 95% confidence intervals (primary model)
if fit_main is not None:
    ci = fit_main.conf_int()
    fixed_effects = pd.DataFrame(
        {
            "Coefficient": fit_main.params,
            "Std Error": fit_main.bse,
            "z value": fit_main.tvalues,
            "p value": fit_main.pvalues,
            "CI Lower": ci[0],
            "CI Upper": ci[1],
        }
    )
    print("\n=== FIXED EFFECTS TABLE (PRIMARY MODEL) ===")
    print(fixed_effects.round(4))


# 3. Residual Diagnostics & Assumption Checks
def plot_model_values(fitted_values, residuals, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Residuals vs Fitted
    axes[0].scatter(fitted_values, residuals, alpha=0.6, color="teal")
    axes[0].axhline(0, color="red", linestyle="--", linewidth=1.5)
    axes[0].set_title(f"{title}: Residuals vs. Fitted Values")
    axes[0].set_xlabel("Fitted Values (Predicted RPE)")
    axes[0].set_ylabel("Residuals")

    # Q-Q Plot
    stats.probplot(residuals, dist="norm", plot=axes[1])
    axes[1].set_title(f"{title}: Normal Q-Q Plot")

    plt.tight_layout()
    save_fig(fig, f"fig14_lmm_diagnostics_{title.split()[0].lower()}")
    plt.show()


for name, model in models.items():
    plot_model_values(model.fittedvalues, model.resid, title=name)


In [ ]:
def plot_trajectories(sessions):
    d = sessions[sessions["completed"]]
    panels = [("rpe", "Borg RPE (6-20)"), ("mental", "Mental demand (0-100)")]
    fig, axes = plt.subplots(2, 1, figsize=(11, 6.6), sharex=True)
    n = N_WEEKS * SESSIONS_PER_WEEK
    for ax, (col, label) in zip(axes, panels):
        for cond in CONDITIONS:
            g = d[d["condition"] == cond].groupby("session_index")[col]
            m, se = g.mean(), g.std() / np.sqrt(g.count())
            ax.fill_between(m.index.astype(float), m - se, m + se, color=COND_COLORS[cond], alpha=0.14, lw=0)
            ax.plot(m.index.astype(float), m, color=COND_COLORS[cond], lw=2, marker="o", ms=5, label=cond)
            ax.text(n + 0.3, m.iloc[-1], cond, color=COND_COLORS[cond], va="center", fontsize=9, fontweight="bold")
        for w in range(1, N_WEEKS):
            ax.axvline(w * SESSIONS_PER_WEEK + 0.5, color=AXIS, lw=1)
        ax.set_ylabel(label)
        ax.set_xlim(0.5, n + 2.4)
    short = {"Easy run": "Easy", "Interval run": "Int.", "Tempo run": "Tempo", "Long run": "Long"}
    axes[1].set_xticks(range(1, n + 1), [f"{short[PLAN[(k - 1) % 5]]}" for k in range(1, n + 1)], fontsize=8)
    for w in range(N_WEEKS):
        axes[0].text(w * SESSIONS_PER_WEEK + 3, 1.02, f"Week {w + 1}", transform=axes[0].get_xaxis_transform(),
                     ha="center", color=INK2, fontsize=9)
    axes[0].legend(loc="upper left", ncol=2, fontsize=9)
    axes[0].set_title("RQ1 + RQ2 · Session-by-session ratings across the plan (mean ± 1 SE)", pad=18)
    fig.tight_layout()
    save_fig(fig, "fig04_trajectories")
    return fig


plot_trajectories(sessions);

### Estimation plots (Gardner-Altman)
Instead of a bar chart with a p-value, each panel shows **every runner's average** (left) and the **difference between conditions with its bootstrap 95% CI** (right; the shaded shape is the bootstrap distribution). This shows both the size of the gap and how uncertain it is with six runners per group.

In [ ]:
def plot_estimation(pm, outcomes):
    fig = plt.figure(figsize=(14, 4.4))
    outer = fig.add_gridspec(1, len(outcomes), wspace=0.62)
    g = np.random.default_rng(SEED)
    results = {}
    for k, (col, label) in enumerate(outcomes.items()):
        inner = outer[k].subgridspec(1, 2, width_ratios=[1.6, 1], wspace=0.08)
        ax, axd = fig.add_subplot(inner[0]), fig.add_subplot(inner[1])
        res = bootstrap_diff(pm, col)
        results[col] = res
        for i, cond in enumerate(CONDITIONS):
            v = pm.loc[pm["condition"] == cond, col].dropna().to_numpy()
            ax.scatter(i + g.uniform(-0.12, 0.12, len(v)), v, s=42, color=COND_COLORS[cond], edgecolor=SURFACE,
                       linewidth=1, zorder=3)
            boots = [g.choice(v, len(v)).mean() for _ in range(2000)]
            lo, hi = np.percentile(boots, [2.5, 97.5])
            ax.plot([i + 0.27] * 2, [lo, hi], color=INK, lw=1.6)
            ax.scatter(i + 0.27, v.mean(), s=30, color=INK, zorder=4)
        ax.set_xticks([0, 1], ["Full\nPlan", "Current\nOnly"])
        ax.set_xlim(-0.45, 1.5)
        ax.set_ylabel(label)
        ax.grid(axis="x", visible=False)
        ax.set_title(label.split(" (")[0], fontsize=11)
        parts = axd.violinplot(res["boot"], positions=[0], widths=0.7, showextrema=False)
        for b in parts["bodies"]:
            b.set_facecolor(MUTED); b.set_alpha(0.25); b.set_edgecolor("none")
        axd.plot([0, 0], [res["diff_lo"], res["diff_hi"]], color=INK, lw=1.8)
        axd.scatter(0, res["diff"], s=45, color=INK, zorder=3)
        axd.axhline(0, color=INK2, lw=1)
        axd.yaxis.tick_right(); axd.yaxis.set_label_position("right")
        axd.spines["right"].set_visible(True); axd.spines["left"].set_visible(False)
        axd.set_ylabel("FP − CO difference")
        axd.set_xticks([0], ["Δ"])
        axd.set_xlim(-0.6, 0.6)
        axd.grid(axis="x", visible=False)
        axd.set_title(f"Δ {res['diff']:+.1f}\n[{res['diff_lo']:+.1f}, {res['diff_hi']:+.1f}]", fontsize=9,
                      fontweight="normal", color=INK2, loc="center")
    fig.suptitle("RQ1 + RQ2 · Each runner's average, and the Full Plan − Current Only difference (bootstrap 95% CI)",
                 x=0.06, ha="left", fontweight="bold", fontsize=12, y=1.04)
    save_fig(fig, "fig05_estimation_plots")
    return fig, results


_, est_results = plot_estimation(pm_all, OUTCOMES)

### Anticipatory pacing: does knowing what comes next change effort?
Anticipatory regulation (Tucker, 2009) predicts that runners who **can see** the next session adjust today's effort: holding back before a harder session, pushing before an easy one or on the final session. Current Only runners cannot see ahead, so their pattern should be flat.

To isolate this, each rating is adjusted for the runner's usual level and the session type's usual level (`rpe_adj`, `mental_adj`), so what remains is "harder or easier than expected for this runner on this kind of session". Dots are runner averages; bars are 95% CIs across runners.

In [ ]:
def plot_anticipatory(sessions):
    d = sessions[sessions["completed"]]
    cats = list(d["next_category"].cat.categories)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=False)
    g = np.random.default_rng(SEED)
    out = []
    for ax, (col, label) in zip(axes, [("rpe_adj", "RPE vs. expected (points)"), ("mental_adj", "Mental demand vs. expected (0-100 points)")]):
        rm = d.groupby(["participant_code", "condition", "next_category"], observed=True)[col].mean().reset_index()
        for k, cond in enumerate(CONDITIONS):
            off = -0.17 if k == 0 else 0.17
            for i, cat in enumerate(cats):
                v = rm.loc[(rm["condition"] == cond) & (rm["next_category"] == cat), col].dropna()
                if len(v) == 0:
                    continue
                ax.scatter(i + off + g.uniform(-0.05, 0.05, len(v)), v, s=18, color=COND_COLORS[cond], alpha=0.5, lw=0)
                # a CI from fewer than 3 runners is too wide to be informative, so it is not drawn
                half = stats.t.ppf(0.975, len(v) - 1) * v.std() / np.sqrt(len(v)) if len(v) >= 3 else np.nan
                ax.errorbar(i + off, v.mean(), yerr=None if np.isnan(half) else half, fmt="o", ms=8,
                            color=COND_COLORS[cond], mec=SURFACE, mew=1.5, elinewidth=1.8, capsize=0,
                            label=cond if i == 0 else None)
                if len(v) < 3:
                    ax.annotate(f"{len(v)} runners,\nno CI", (i + off, v.mean()), xytext=(0, -26),
                                textcoords="offset points", ha="center", fontsize=7.5, color=MUTED)
                out.append(dict(measure=col, condition=cond, next=cat, mean=v.mean(), ci=half, runners=len(v)))
        ax.axhline(0, color=INK2, lw=1)
        ax.set_xticks(range(len(cats)), [c.capitalize() for c in cats])
        ax.set_ylabel(label)
        ax.grid(axis="x", visible=False)
    axes[0].legend(loc="upper left", fontsize=9)
    axes[0].set_title("RQ2 · Effort relative to expected, by what comes next")
    axes[1].set_title("RQ1 · Mental demand relative to expected")
    fig.tight_layout()
    save_fig(fig, "fig06_anticipatory_pacing")
    return fig, pd.DataFrame(out)


_, anticip = plot_anticipatory(sessions)
anticip.pivot_table(index=["measure", "next"], columns="condition", values="mean", observed=True).round(2)

### Exploratory inferential checks
Not part of the planned descriptive analysis and **not confirmatory** (8 runners with data: 5 Full Plan, 3 Current Only). They show whether the descriptive differences are larger than what random assignment alone would produce, and whether the mixed model (using every session) agrees with the runner-level comparison.

In [ ]:
rows = []
for col, label in OUTCOMES.items():
    perm = stratified_permutation_test(pm_all, col)
    est = est_results[col]
    mm = fit_mixed(sessions, col)
    rows.append({"outcome": label,
                 "Full Plan mean": pm_all.loc[pm_all.condition == "Full Plan", col].mean(),
                 "Current Only mean": pm_all.loc[pm_all.condition == "Current Only", col].mean(),
                 "difference [95% CI]": f"{est['diff']:+.2f} [{est['diff_lo']:+.2f}, {est['diff_hi']:+.2f}]",
                 "Hedges g [95% CI]": f"{est['g']:.2f} [{est['g_lo']:.2f}, {est['g_hi']:.2f}]",
                 "permutation p": perm["p"], "assignments": perm["n_assignments"],
                 "mixed model FP effect [95% CI]": f"{mm['estimate']:+.2f} [{mm['ci_lo']:+.2f}, {mm['ci_hi']:+.2f}]",
                 "mixed model p": mm["p"]})
inferential = pd.DataFrame(rows)
inferential.insert(inferential.columns.get_loc("assignments"), "Holm-adjusted p", holm(inferential["permutation p"]))
inferential.round(3)

## 9. RQ3: Cognitive load and perceived exertion
Mental demand is used instead of overall TLX, because the physical demand and effort subscales overlap with RPE by definition. The relationship is looked at on **two levels that answer different questions**:

* **Within runners** (repeated-measures correlation): on sessions where a runner's mental demand is higher than *their own* usual, is their RPE also higher?
* **Between runners** (Spearman on averages): do runners with higher average mental demand report higher average RPE?

The proposal's session-level Spearman is also shown, with the caveat that it treats repeated sessions from one runner as independent.

In [ ]:
done = sessions[sessions["completed"]]
corr_rows = []
for scope in ["All runners", *CONDITIONS]:
    d = done if scope == "All runners" else done[done["condition"] == scope]
    pmd = pm_all if scope == "All runners" else pm_all[pm_all["condition"] == scope]
    rm = rmcorr(d, "mental", "rpe")
    lo, hi = rmcorr_boot_ci(d, "mental", "rpe")
    b = stats.spearmanr(pmd["mental"], pmd["rpe"])
    sl = stats.spearmanr(d["mental"], d["rpe"], nan_policy="omit")
    corr_rows.append({"scope": scope,
                      "within-runner r_rm [95% CI]": f"{rm['r']:.2f} [{lo:.2f}, {hi:.2f}]", "r_rm df": rm["dof"],
                      "between-runner Spearman rho": round(b.statistic, 2), "runners": len(pmd),
                      "session-level Spearman rho (proposal)": round(sl.statistic, 2), "sessions": rm["n_obs"]})
pd.DataFrame(corr_rows)

In [ ]:
def plot_rq3(sessions):
    d = sessions[sessions["completed"]].dropna(subset=["mental", "rpe"])
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6), sharex=True, sharey=True)
    g = np.random.default_rng(SEED)
    for ax, cond in zip(axes, CONDITIONS):
        dc = d[d["condition"] == cond]
        ax.scatter(dc["mental"] + g.uniform(-1.5, 1.5, len(dc)), dc["rpe"] + g.uniform(-0.15, 0.15, len(dc)),
                   s=16, color=COND_COLORS[cond], alpha=0.35, lw=0)
        for _, grp in dc.groupby("participant_code"):
            if grp["mental"].nunique() < 2:
                continue
            b1, b0 = np.polyfit(grp["mental"], grp["rpe"], 1)
            xs = np.array([grp["mental"].min(), grp["mental"].max()])
            ax.plot(xs, b0 + b1 * xs, color=COND_COLORS[cond], lw=1.4, alpha=0.8)
        rm = rmcorr(dc, "mental", "rpe")
        ax.set_title(f"{cond}:  within-runner r_rm = {rm['r']:.2f}", color=INK)
        ax.set_xlabel("Mental demand (0-100)")
    axes[0].set_ylabel("Borg RPE (6-20)")
    axes[0].legend(handles=[Line2D([], [], color=INK2, lw=1.4, label="one runner's trend"),
                            Line2D([], [], marker="o", ls="", color=MUTED, alpha=.5, label="one session")],
                   loc="upper left", fontsize=9)
    fig.suptitle("RQ3 · Mental demand and RPE within each runner", x=0.06, ha="left", fontweight="bold", y=1.0)
    fig.tight_layout()
    save_fig(fig, "fig07_rq3_within_runner")
    return fig


plot_rq3(sessions);

In [ ]:
def plot_runner_forest(sessions):
    pr = per_runner_spearman(sessions[sessions["completed"]], "mental", "rpe")
    pr["condition"] = pd.Categorical(pr["condition"], CONDITIONS)
    pr = pr.sort_values(["condition", "rho"], ascending=[False, True]).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, 0.38 * len(pr) + 1.4))
    for i, r in pr.iterrows():
        ax.plot([r["lo"], r["hi"]], [i, i], color=COND_COLORS[r["condition"]], lw=1.8, alpha=0.6)
        ax.scatter(r["rho"], i, s=50, color=COND_COLORS[r["condition"]], edgecolor=SURFACE, lw=1.2, zorder=3)
        ax.text(1.04, i, f"n = {r['n']}", va="center", fontsize=8, color=MUTED)
    ax.axvline(0, color=INK2, lw=1)
    ax.set_yticks(range(len(pr)), pr["participant_code"])
    ax.set_xlim(-1, 1)
    ax.set_xlabel("Spearman rho between mental demand and RPE across that runner's sessions (95% CI)")
    ax.grid(axis="y", visible=False)
    ax.legend(handles=[Line2D([], [], marker="o", ls="", ms=7, color=COND_COLORS[c], label=c) for c in CONDITIONS],
              loc="lower right", fontsize=9)
    ax.set_title("RQ3 · Does the link hold for each runner?")
    fig.tight_layout()
    save_fig(fig, "fig08_rq3_per_runner")
    return fig, pr


_, per_runner = plot_runner_forest(sessions)

## 10. Content analysis of the diary answers (RQ1–RQ3)
### 10a. Codebook
| Diary question | RQ | What the codes capture |
|---|---|---|
| Did what you knew about upcoming sessions affect how hard you pushed? | RQ2 | Pacing decision (prepared / held back / pushed / no effect / unsure), and whether it drew on knowing or not knowing what was next, or on today's session |
| What did you have to think about or keep track of? | RQ1 | Sources of cognitive load (time and pace, workout structure, the body, following the plan, outside demands, finding the plan, looking ahead) |
| Did anything on your mind about the plan make the run easier or harder? | RQ3 | Direction of the effect and the mechanism (session framing, outside demands, the plan as motivation, social, anticipation, surprise) |

In [ ]:
codebook_table = pd.DataFrame([
    {"question": q, "type": "one per answer" if kind == "single" else "zero or more",
     "code": code, "definition": desc, "rule pattern": pat}
    for q, book in CODEBOOK.items() for kind in ("single", "multi") for code, (desc, pat) in book[kind].items()])
codebook_table.to_csv(OUT_DIR / "codebook.csv", index=False)
codebook_table[["question", "type", "code", "definition"]]

### 10b. Coding: rule-based and LLM

In [ ]:
rule_codes = code_responses_rules(text_responses)
llm_codes, llm_status = run_llm_coding(text_responses)
print(f"Rule-based coder: {len(rule_codes)} answers coded.")
print(f"LLM coder ({LLM_PROVIDER}): {len(llm_codes)} answers coded. Prompt files are in {LLM_DIR}/")
llm_status

**Coder agreement.** When LLM codes exist, each code is compared with the rule-based coder using Cohen's κ (≥ 0.61 substantial, ≥ 0.81 almost perfect; Landis & Koch, 1977). Codes with low agreement are reviewed by hand before use. A stratified **human validation sample** (25% of answers, blind to condition) is exported for two team members to code; once `human_validation_coded.csv` is filled in, the same table also reports human-vs-LLM agreement.

In [ ]:
def agreement_table(a, b, name_a, name_b):
    m = a.merge(b, on=["response_id", "question"], suffixes=("_a", "_b"))
    rows = []
    for q, book in CODEBOOK.items():
        mq = m[m["question"] == q]
        cols = ([book["single_name"]] if book["single_name"] else []) + list(book["multi"])
        for c in cols:
            if len(mq) and f"{c}_a" in mq:
                rows.append({"comparison": f"{name_a} vs {name_b}", "question": q, "code": c, "answers": len(mq),
                             "% agree": 100 * np.mean(mq[f"{c}_a"].astype(str) == mq[f"{c}_b"].astype(str)),
                             "kappa": cohen_kappa(mq[f"{c}_a"], mq[f"{c}_b"])})
    return pd.DataFrame(rows)


sample = (text_responses[text_responses["question"].isin(CODED_QUESTIONS)]
          .groupby(["question", "condition"], observed=True).sample(frac=0.25, random_state=SEED))
human_file = OUT_DIR / "human_validation_sample.csv"
CODE_COLUMNS = list(dict.fromkeys(c for book in CODEBOOK.values()
                                  for c in ([book["single_name"]] if book["single_name"] else []) + list(book["multi"])))
template = sample[["response_id", "question", "response"]].sample(frac=1, random_state=SEED).reset_index(drop=True)
for c in CODE_COLUMNS:
    template[c] = ""
# Rewrite the blank template whenever it does not match the current answers (e.g. it was made from the synthetic file)
current = pd.read_csv(human_file, dtype=str, keep_default_na=False) if human_file.exists() else None
if current is None or current[["response_id", "response"]].values.tolist() != template[["response_id", "response"]].values.tolist():
    template.to_csv(human_file, index=False)
print(f"Human validation sample: {len(template)} answers in {human_file}. A team member copies it to "
      f"human_validation_coded.csv and fills, for each row, the single label ({', '.join(b['single_name'] for b in CODEBOOK.values() if b['single_name'])}) "
      "and 1/0 for each theme of that row's question (other questions' columns stay blank). The coder sees no condition or participant.")

agreement = []
if len(llm_codes):
    agreement.append(agreement_table(rule_codes, llm_codes, "rules", "LLM"))
coded_human = OUT_DIR / "human_validation_coded.csv"
if coded_human.exists():
    human = pd.read_csv(coded_human, dtype=str, keep_default_na=False)
    for q, book in CODEBOOK.items():        # 1/0, yes/no, TRUE/FALSE all become booleans, like the coders' output
        for c in book["multi"]:
            human.loc[human["question"] == q, c] = human.loc[human["question"] == q, c].str.strip().str.lower().isin(["1", "true", "yes", "y", "x"])
    if len(llm_codes):
        agreement.append(agreement_table(human, llm_codes, "human", "LLM"))
    agreement.append(agreement_table(human, rule_codes, "human", "rules"))
if agreement:
    display_agreement = pd.concat(agreement).round(2)
else:
    display_agreement = pd.DataFrame({"status": [f"No human codes yet. Fill in {coded_human.name} (see above) and re-run "
                                                 "to get human-vs-rules agreement (Cohen's kappa)."]})
display_agreement

In [ ]:
use_llm = len(llm_codes) and llm_status["problems"].eq("").all()
codes = (llm_codes if use_llm else rule_codes).drop(columns="evidence", errors="ignore")
CODE_SOURCE = "LLM" if use_llm else "rule-based"
coded = text_responses.merge(codes, on=["response_id", "question"], how="inner")
coded.to_csv(OUT_DIR / "coded_responses.csv", index=False)
print(f"Analysis below uses {CODE_SOURCE} codes for {len(coded)} answers.")
uncoded = coded[(coded.get("pacing", pd.Series(dtype=str)) == "UNCODED") | (coded.get("valence", pd.Series(dtype=str)) == "UNCODED")]
print(f"Answers the coder could not place (review by hand): {len(uncoded)}")
uncoded[["response_id", "question", "response"]].head(10)

### 10c. Manipulation check from the diary text
If the manipulation worked, Full Plan runners should refer to upcoming sessions and the rest of the plan, while Current Only runners should mention not knowing what comes next and having to find each session's message. Both groups are also expected to reason about *today's* session; that row is the baseline. This complements the interview manipulation check.

In [ ]:
def entry_level(coded):
    flags = {
        "used knowledge of upcoming sessions": ["KNEW_UPCOMING", "LOOK_AHEAD", "ANTICIPATING"],
        "mentioned not knowing what comes next": ["UNCERTAIN_UPCOMING", "SURPRISE"],
        "based effort or feelings on today's session": ["TODAY_SESSION", "SESSION_FRAMING"],
        "had to find or check the session": ["PLAN_ACCESS"],
        "focused only on today": ["FOCUS_TODAY"],
    }
    e = coded.copy()
    for name, cols in flags.items():
        e[name] = e[[c for c in cols if c in e]].fillna(False).astype(bool).any(axis=1)
    per_entry = e.groupby(["participant_code", "condition", "session_index"], observed=True)[list(flags)].any().reset_index()
    return per_entry, list(flags)


entries, check_cols = entry_level(coded)
manip = entries.groupby("condition", observed=True)[check_cols].mean().T * 100
manip["runners with ≥1 mention (FP / CO)"] = [
    f"{entries[entries.condition == 'Full Plan'].groupby('participant_code')[c].any().sum()} / "
    f"{entries[entries.condition == 'Current Only'].groupby('participant_code')[c].any().sum()}" for c in check_cols]
manip.round(1).rename(columns={c: f"% of entries · {c}" for c in CONDITIONS})

### 10d. Pacing decisions (RQ2): what runners said they did, by what came next
A diverging bar: **held back** extends left, **pushed** extends right, neutral answers sit in the middle. If anticipatory regulation is at work, Full Plan runners should hold back before harder sessions and push before easy runs and the final session; Current Only rows should look alike.

In [ ]:
def plot_pacing_diverging(coded):
    d = coded[coded["question"] == "q_pacing"].copy()
    d["pacing"] = d["pacing"].replace({"UNCODED": "UNSURE"})
    cats = list(sessions["next_category"].cat.categories)
    rows = [(c, n) for c in CONDITIONS for n in cats]
    pct = (d.groupby(["condition", "next_category"], observed=True)["pacing"].value_counts(normalize=True)
           .unstack(fill_value=0).reindex(columns=["HELD_BACK", "PREPARED", "NO_EFFECT", "UNSURE", "PUSHED"], fill_value=0) * 100)
    counts = d.groupby(["condition", "next_category"], observed=True).size()
    colors = {"HELD_BACK": "#6250d6", "PREPARED": "#2a9d8f", "NO_EFFECT": "#c3c2b7", "UNSURE": "#e1e0d9", "PUSHED": "#e34948"}
    labels = {"HELD_BACK": "Held back", "PREPARED": "Prepared", "NO_EFFECT": "No effect", "UNSURE": "Unsure", "PUSHED": "Pushed harder"}
    fig, ax = plt.subplots(figsize=(11, 4.4))
    ylabels = []
    for i, (cond, nxt) in enumerate(rows):
        y = len(rows) - 1 - i + (0.5 if cond == "Full Plan" else 0)
        if (cond, nxt) not in pct.index:
            continue
        p = pct.loc[(cond, nxt)]
        neutral = p["PREPARED"] + p["NO_EFFECT"] + p["UNSURE"]
        segs = [("HELD_BACK", -neutral / 2 - p["HELD_BACK"], p["HELD_BACK"]),
                ("PREPARED", -neutral / 2, p["PREPARED"]),
                ("NO_EFFECT", -neutral / 2 + p["PREPARED"], p["NO_EFFECT"]),
                ("UNSURE", -neutral / 2 + p["PREPARED"] + p["NO_EFFECT"], p["UNSURE"]),
                ("PUSHED", neutral / 2, p["PUSHED"])]
        for code, left, width in segs:
            if width <= 0:
                continue
            ax.barh(y, width, left=left, height=0.72, color=colors[code], edgecolor=SURFACE, linewidth=2)
            if width >= 9:
                ax.text(left + width / 2, y, f"{width:.0f}%", ha="center", va="center", fontsize=8,
                        color="white" if code in ("HELD_BACK", "PREPARED", "PUSHED") else INK)
        ax.text(102, y, f"n = {counts.loc[(cond, nxt)]}", va="center", fontsize=8, color=MUTED)
        ylabels.append((y, nxt.capitalize(), cond))
    ax.set_yticks([y for y, *_ in ylabels], [l for _, l, _ in ylabels])
    for cond in CONDITIONS:
        ys = [y for y, _, c in ylabels if c == cond]
        ax.annotate(cond, (-0.29, np.mean(ys)), xycoords=("axes fraction", "data"), rotation=90, ha="center",
                    va="center", color=COND_COLORS[cond], fontweight="bold", annotation_clip=False)
    ax.axvline(0, color=INK2, lw=1)
    ax.set_xlim(-100, 110)
    ax.set_xticks(range(-100, 101, 25), [f"{abs(v)}%" for v in range(-100, 101, 25)])
    ax.set_xlabel("← held back        % of diary answers        pushed harder →")
    ax.grid(axis="y", visible=False)
    ax.legend(handles=[Patch(facecolor=colors[c], label=labels[c]) for c in colors],
              loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=5, fontsize=9)
    ax.set_title(f"RQ2 · Self-reported pacing decisions by what came next ({CODE_SOURCE} codes)")
    fig.tight_layout()
    fig.subplots_adjust(left=0.2)
    save_fig(fig, "fig09_pacing_diverging")
    return fig


plot_pacing_diverging(coded);

### 10e. How themes change over the plan
Share of answers mentioning each theme, by condition and week. Look for themes that belong to one condition (e.g., *Uncertainty* in Current Only) and themes that grow toward the end of the plan (e.g., *Progress* in Full Plan).

In [ ]:
def plot_theme_heatmap(coded):
    themes = [(q, c) for q in CODED_QUESTIONS for c in CODEBOOK[q]["multi"]]
    qlabel = {"q_pacing": "Pacing", "q_tracking": "Load source", "q_plan_mind": "Plan effect"}
    mats = {}
    for cond in CONDITIONS:
        m = np.full((len(themes), N_WEEKS), np.nan)
        for i, (q, c) in enumerate(themes):
            sub = coded[(coded["condition"] == cond) & (coded["question"] == q)]
            m[i] = sub.groupby("week")[c].mean().reindex(range(1, N_WEEKS + 1)).to_numpy(float) * 100
        mats[cond] = m
    vmax = max(np.nanmax(m) for m in mats.values())
    cmap = LinearSegmentedColormap.from_list("seq", ["#f4f8fd", "#9ec5f4", "#3987e5", "#184f95", "#0d366b"])
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 0.36 * len(themes) + 1.8), sharey=True)
    for ax, cond in zip(axes, CONDITIONS):
        im = ax.imshow(mats[cond], cmap=cmap, vmin=0, vmax=vmax, aspect="auto")
        for i in range(len(themes)):
            for j in range(N_WEEKS):
                v = mats[cond][i, j]
                if np.isfinite(v):
                    ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8,
                            color="white" if v > vmax * 0.55 else INK)
        ax.set_xticks(range(N_WEEKS), [f"Week {w}" for w in range(1, N_WEEKS + 1)])
        ax.set_title(cond, color=COND_COLORS[cond])
        ax.grid(False)
        ax.set_xticks(np.arange(-.5, N_WEEKS, 1), minor=True)
        ax.set_yticks(np.arange(-.5, len(themes), 1), minor=True)
        ax.grid(which="minor", color=SURFACE, lw=2)
        ax.tick_params(which="both", length=0)
        for sp in ax.spines.values():
            sp.set_visible(False)
    axes[0].set_yticks(range(len(themes)), [f"{qlabel[q]} · {c.replace('_', ' ').title()}" for q, c in themes], fontsize=9)
    cb = fig.colorbar(im, ax=axes, shrink=0.6, pad=0.02)
    cb.set_label("% of answers to that question")
    cb.outline.set_visible(False)
    fig.suptitle(f"Themes by condition and week ({CODE_SOURCE} codes)", x=0.02, ha="left", fontweight="bold")
    save_fig(fig, "fig10_theme_heatmap")
    return fig


plot_theme_heatmap(coded);

### 10f. Example quotes per theme
For the write-up, a few answers per theme and condition (labelled by participant code and session only). Check every quote for identifying details before using it.

In [ ]:
def exemplar_quotes(coded, per_cell=2):
    rows = []
    for q in CODED_QUESTIONS:
        for c in CODEBOOK[q]["multi"]:
            for cond in CONDITIONS:
                sub = coded[(coded["question"] == q) & (coded["condition"] == cond) & coded[c].fillna(False).astype(bool)
                            & coded["substantive"]]
                sub = sub.drop_duplicates("participant_code").sample(frac=1, random_state=SEED).head(per_cell)
                for _, r in sub.iterrows():
                    rows.append({"theme": c, "condition": cond, "who": f"{r['participant_code']} · W{r['week']} S{r['session']}",
                                 "quote": r["response"]})
    return pd.DataFrame(rows)


quotes = exemplar_quotes(coded)
quotes.to_csv(OUT_DIR / "exemplar_quotes.csv", index=False)
quotes.head(24)

## 11. Sensitivity analysis
Do the condition differences depend on cleaning decisions? Each row re-computes the runner-level difference after removing one category of questionable data. A finding that changes direction across rows should not be reported without that caveat.

In [ ]:
SCENARIOS = {
    "All completed sessions": None,
    "Excluding late entries (> 2 h)": ~sessions["flag_late"],
    "Sessions run as prescribed only": sessions["status"].eq("prescribed"),
    "Excluding any quality flag": ~sessions["any_quality_flag"],
    "Runners who logged to the end only": ~sessions["participant_code"].isin(
        participants.loc[participants["stopped_early"] | participants["withdrawn"], "participant_code"]),
    f"Excluding runners below {MIN_SUBMISSION_RATE:.0%} of entries (proposal's rule)": ~sessions["participant_code"].isin(
        participants.loc[participants["below_min_submission"].fillna(False).astype(bool), "participant_code"]),
}
print("Stopped logging early:", ", ".join(participants.loc[participants["stopped_early"] & ~participants["did_not_start"], "participant_code"]))
print(f"Below {MIN_SUBMISSION_RATE:.0%} of entries:",
      ", ".join(participants.loc[participants["below_min_submission"].fillna(False).astype(bool), "participant_code"]))
sens = []
for name, mask in SCENARIOS.items():
    pm = participant_means(sessions, list(OUTCOMES), mask)
    for col, label in OUTCOMES.items():
        a = pm.loc[pm.condition == "Full Plan", col].dropna()
        b = pm.loc[pm.condition == "Current Only", col].dropna()
        sens.append({"scenario": name, "outcome": label, "sessions used": int(sessions["completed"].sum() if mask is None
                     else (sessions["completed"] & mask).sum()), "FP mean": a.mean(), "CO mean": b.mean(),
                     "difference": a.mean() - b.mean(), "Hedges g": hedges_g(a, b)})
sensitivity = pd.DataFrame(sens)
sensitivity.pivot(index="scenario", columns="outcome", values="difference").loc[list(SCENARIOS)].round(2)

### Stability of Borg RPE using Leave-One-Out (LOO)

Leave-One-Out process is used to help determine whether if a particular runner has an unusually large influence on the mean RPE, separated by condition.

In [ ]:
# 1. Identify unique participants
loo_participants = rpe_df["participant_code"].dropna().unique()

loo_records = []

# 2. Iterate and drop one participant at a time
for p in loo_participants:
  subset = rpe_df[rpe_df["participant_code"] != p]

  # Calculate summary stats for the remaining sample
  loo_stats = (
      subset.groupby("condition")["rpe"]
      .agg(
          Mean="mean",
          Std="std",
          Median="median",
          IQR=lambda x: x.quantile(0.75) - x.quantile(0.25),
          Count="count",
      )
      .reset_index()
  )

  loo_stats["dropped_participant"] = str(p)
  loo_records.append(loo_stats)

# Combine into a single DataFrame
loo_df = pd.concat(loo_records, ignore_index=True)

# 3. Get overall Stability Summary Across LOO Iterations
stability_summary = (
    loo_df.groupby("condition")["Mean"]
    .agg(
        Baseline_LOO_Mean="mean",
        Min_Mean="min",
        Max_Mean="max",
        Max_Range=lambda x: x.max() - x.min(),  # Total fluctuation span
        Std_of_Means="std",  # Lower = more stable
    )
    .reset_index()
)

print("=== LEAVE-ONE-OUT SENSITIVITY SUMMARY ===")
print(stability_summary)

# 4. Plot LOO Stability Trajectory
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 5))

# Plot mean RPE for each condition as participants are dropped one by one
ax = sns.lineplot(
    data=loo_df,
    x="dropped_participant",
    y="Mean",
    hue="condition",
    marker="o",
    palette="Set2",
    linewidth=0,
    markersize=7,
)

plt.title(
    "Leave-One-Participant-Out Sensitivity: Mean RPE Stability",
    fontsize=13,
    pad=12,
)
plt.xlabel("Dropped Participant Code", fontsize=11)
plt.ylabel("Condition Mean RPE", fontsize=11)
plt.xticks(rotation=45)
plt.legend(title="Condition", loc="best")

plt.tight_layout()
save_fig(plt.gcf(), "fig15_loo_rpe")
plt.show()

In [ ]:
# Pivot to compute condition difference (e.g., Condition A - Condition B) per iteration
loo_pivot = loo_df.pivot(
    index="dropped_participant", columns="condition", values="Mean"
)

# Use current only as baseline
if len(loo_pivot.columns) >= 2:
  cond_a, cond_b = loo_pivot.columns[1], loo_pivot.columns[0]
  loo_pivot["Mean_Difference"] = loo_pivot[cond_a] - loo_pivot[cond_b]

  print(f"\n=== MEAN DIFFERENCE ({cond_a} vs {cond_b}) STABILITY ===")
  print(loo_pivot[["Mean_Difference"]].describe())

### Sensitivity Analysis using different exclusion scenarios

Filtering rules are applied to the dataset:
- **All completed sessions**: Includes all data without filtering.
- **Excluding late entries (> 2 h)**: Excludes sessions flagged as late `(~sessions["flag_late"])`.
- **Sessions run as prescribed only**: Keeps only sessions where execution matched protocol `(status == "prescribed")`.
- **Excluding any quality flag**: Filters out any session flagged for quality issues `(~sessions["any_quality_flag"]`).
- **Runners who logged to the end only**: Excludes runners who stopped logging early (3 or more unlogged sessions at the end) or confirmed a withdrawal.
- **Excluding runners below 50% of entries**: Applies the proposal's exclusion rule (fewer than 50% of diary entries submitted), which the main analysis does not apply.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Scenario order
scenario_order = list(SCENARIOS.keys())

outcome_order = [
    "Borg RPE (6-20)",
    "Mental demand (0-100)",
    "Raw TLX (0-100)"
]

short_labels = [
    "All completed sessions",
    "Exclude late entries",
    "Prescribed sessions only",
    "Exclude quality flags",
    "Logged to the end only",
    "Exclude < 50% of entries"
]

colors = ["#2878B5", "#E07A2D", "#31966B"]

sns.set_theme(style="whitegrid", context="paper")

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 6.5)
)

for ax, outcome, color in zip(
    axes, outcome_order, colors
):

    subset = (
        sensitivity[
            sensitivity["outcome"] == outcome
        ]
        .set_index("scenario")
        .loc[scenario_order]
    )

    differences = subset["difference"].values
    baseline = differences[0]

    y = np.arange(len(scenario_order))

    # Plot baseline reference line
    ax.axvline(
        baseline,
        color="black",
        linestyle="--",
        linewidth=1.5,
        label=f"Baseline = {baseline:.2f}"
    )

    # Plot scenario estimates
    ax.scatter(
        differences,
        y,
        color=color,
        s=85,
        zorder=3,
        edgecolor="white",
        linewidth=0.8
    )

    # Numerical labels
    for x, ypos in zip(differences, y):
        ax.annotate(
            f"{x:.2f}",
            (x, ypos),
            xytext=(7, 0),
            textcoords="offset points",
            va="center",
            fontsize=9
        )

    ax.set_yticks(y)
    ax.set_yticklabels(short_labels)

    ax.invert_yaxis()

    ax.set_title(
        outcome,
        fontsize=12,
        fontweight="bold"
    )

    ax.set_xlabel("Mean difference (FP - CO)")

    ax.grid(
        axis="x",
        alpha=0.3
    )

    ax.grid(
        axis="y",
        visible=False
    )

    ax.legend(
        frameon=False,
        fontsize=8,
        loc="lower right"
    )

fig.suptitle(
    "Sensitivity Analysis: Differences Relative to Baseline",
    fontsize=15,
    fontweight="bold",
    y=1.02
)

plt.tight_layout()
save_fig(fig, "fig16_sensitivity")
plt.show()

### Quantified stability metrics

- `baseline`: The mean difference `(FP - CO)` from the unfiltered dataset
- `max_absolute_deviation`: The maximum shift observed between the baseline and filtered scenario.
- `max_percentage_deviation`: The maximum relative change from baseline expressed as a percentage
- `directional_consistency`: The percentage of scenarios where the mean difference maintains the same algebraic sign as the baseline.

In [ ]:
stability_results = []

for outcome in sensitivity["outcome"].unique():

    sub = sensitivity[
        sensitivity["outcome"] == outcome
    ].copy()

    baseline = sub.loc[
        sub["scenario"] == "All completed sessions",
        "difference"
    ].iloc[0]

    # Exclude baseline when assessing alternative scenarios
    exclusions = sub[
        sub["scenario"] != "All completed sessions"
    ].copy()

    exclusions["change"] = (
        exclusions["difference"] - baseline
    )

    exclusions["percent_change"] = (
        exclusions["change"] / abs(baseline)
    ) * 100

    stability_results.append({
        "outcome": outcome,
        "baseline": baseline,
        "max_absolute_deviation":
            exclusions["change"].abs().max(),
        "max_percentage_deviation":
            exclusions["percent_change"].abs().max(),
        "directional_consistency":
            (
                np.sign(sub["difference"]) == np.sign(baseline)
            ).mean() * 100
    })

stability = pd.DataFrame(stability_results)

display(stability.round(2))


## 12. Summary and export
The cell below prints the headline numbers in one place. **On synthetic data these only confirm that the pipeline recovers the planted effects** (mental demand +15 and RPE +1 for Full Plan, on average; any single sample lands near these values, not on them). The generator writes pacing comments that follow anticipatory regulation, but it does not make the *ratings* follow those comments, so Figure 6 is expected to look flat on this data.

In [ ]:
from scipy import stats

fmt = lambda r: f"{r['diff']:+.2f} [{r['diff_lo']:+.2f}, {r['diff_hi']:+.2f}]"
rq3_all = rmcorr(done, "mental", "rpe")
print("DATA" + (" (SYNTHETIC: not results)" if IS_SYNTHETIC else ""))
print(f"  {participants.shape[0]} runners, {int(sessions['completed'].sum())} completed sessions, "
      f"{int(sessions['status'].eq('skipped').sum())} skipped, {int(sessions['status'].eq('missing').sum())} missing, "
      f"{sessions.loc[sessions.status == 'withdrawn', 'participant_code'].nunique()} confirmed withdrawal(s), "
      f"{int((participants['stopped_early'] & ~participants['did_not_start']).sum())} stopped logging early, "
      f"{int(participants['did_not_start'].sum())} did not start")
print(f"  Tests: {(test_results.result == 'PASS').sum()} passed, {(test_results.result == 'FAIL').sum()} failed")
print("RQ1  Mental demand, Full Plan − Current Only:", fmt(est_results["mental"]), "points (0-100)")
print("     Raw TLX, Full Plan − Current Only:      ", fmt(est_results["tlx_raw"]), "points (0-100)")
print("RQ2  RPE, Full Plan − Current Only:          ", fmt(est_results["rpe"]), "points (6-20)")
print(f"RQ3  Within-runner r_rm (mental, RPE) = {rq3_all['r']:.2f}; between-runner rho = "
      f"{stats.spearmanr(pm_all['mental'], pm_all['rpe']).statistic:.2f}")
print(f"Content analysis uses {CODE_SOURCE} codes; manipulation check (entries using upcoming-session knowledge): "
      + ", ".join(f"{c} {manip.loc['used knowledge of upcoming sessions', c]:.0f}%" for c in CONDITIONS))
print("\nFiles written:")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file() and "llm_coding" not in p.parts: 
        print("  ", p)